# Etapa 1 - Setup do Ambiente e Governança

Nesta etapa inicial, é realizada a preparação completa do ambiente de execução, garantindo a **reprodutibilidade técnica** e a padronização das dependências necessárias para o pipeline.

### Diretrizes e Cuidados de Configuração:

* **Gestão Centralizada de Dependências:** Instalação orientada pelo arquivo `requirements.txt` oficial do repositório, garantindo paridade de versões entre desenvolvimento local e execução para avaliação.
* **Isolamento de Credenciais:** Configuração de variáveis de ambiente (`.env`) e carregamento de Service Accounts GCP via `.gitignore`, impedindo a exposição de chaves privadas em versionamento.
* **Padronização de Estrutura de Storage:** Mapeamento dos caminhos locais da **Arquitetura Medalhão** (`./data_lake/bronze`, `./data_lake/silver`, `./data_lake/gold`), assegurando persistência em formato **Parquet**.
* **Supressão Controlada de Logs:** Ajuste de preferências globais de warnings e formatação de saídas no terminal (Colorama/Tabulate) para manter o pipeline limpo e focado em métricas operacionais.
* **Conectividade Cloud:** Inicialização dos clientes de API e autenticação no **Google BigQuery** para ingestão direta sem downloads intermediários de CSVs.

---
**Saídas desta etapa:** Ambiente validado, diretórios do Data Lake mapeados e bibliotecas carregadas para inicializar a ingestão.

In [6]:
# Instalação das dependências do projeto a partir do requirements.txt oficial,
# garantindo o mesmo ambiente de execução
!pip install -r requirements.txt --quiet --upgrade --no-warn-conflicts

print("Dependências instaladas com sucesso!")

Dependências instaladas com sucesso!


In [7]:
# Imports e configurações globais
import pandas as pd
import numpy as np
import os
import hashlib
from datetime import datetime, timezone
from colorama import Fore, Style, init
from tabulate import tabulate
import warnings
from google.cloud import bigquery
from dotenv import load_dotenv
import glob
 
warnings.filterwarnings('ignore')

init(autoreset=True)

# Configuração de caminhos para simular um Data Lake local
BASE_PATH = "./data_lake"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"
QUALITY_PATH = f"{BASE_PATH}/quality_reports"

for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUALITY_PATH]:
  os.makedirs(path, exist_ok=True)

# Timestamp de ingestão é gerado uma vez aqui e reutilizado em todas as etapas
INGESTION_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print(f"{Fore.GREEN} Ambiente configurado!")
print(f"{Fore.CYAN} Estrutura do Data Lake:")
print(f"    {BASE_PATH}/")
print(f"    ├── bronze/   -> Dados brutos (raw)")
print(f"    ├── silver/  -> Dados limpos e normalizados")
print(f"    ├── gold/    -> Dados agregados para consumo")
print(f"    └── quality_reports/ -> Relatórios de qualidade")
print(f"\n{Fore.YELLOW} Timestamp de ingestão: {INGESTION_TIMESTAMP}")

 Ambiente configurado!
 Estrutura do Data Lake:
    ./data_lake/
    ├── bronze/   -> Dados brutos (raw)
    ├── silver/  -> Dados limpos e normalizados
    ├── gold/    -> Dados agregados para consumo
    └── quality_reports/ -> Relatórios de qualidade

 Timestamp de ingestão: 20261031_212548


# Etapa 2 - Camada BRONZE: Ingestão de Dados Raw

Nesta etapa, o foco é a extração dos dados brutos diretamente das fontes oficiais para a camada **Bronze** do Data Lake local. 

Os dados são armazenados exatamente no formato original em que são lidos — sem qualquer transformação, limpeza ou alteração de tipos —, assegurando **rastreabilidade total** e a capacidade de reprocessamento histórico a partir do dado bruto.

### Decisões Arquiteturais e Diretrizes de Ingestão:

* **Conexão Direta à Nuvem:** Ingestão via consulta direta ao **Google BigQuery**, evitando a manipulação manual de ficheiros CSV estáticos. Isso garante governança, atualização constante da fonte e integridade nos pipelines.
* **Preservação do Histórico:** A gravação local em Parquet adota uma política clara de particionamento e preservação de histórico, garantindo que execuções consecutivas mantenham a linhagem do dado sem sobrescrita destrutiva.
* **Persistência em Parquet:** Armazenamento dos microdados originais (alunos, municípios, UFs e tabelas de metas) em formato columnar Parquet no diretório `./data_lake/bronze`.

---
**Saídas em `data_lake/bronze/`:** Ficheiros Parquet com a cópia exata do BigQuery das 6 entidades obrigatórias (`alunos`, `municipio`, `uf`, `meta_alfabetizacao_brasil`, `meta_alfabetizacao_uf`, `meta_alfabetizacao_municipio`) e da tabela `dicionario`.

In [8]:
# Carrega as configurações do arquivo externo .env
load_dotenv()

# Pega o caminho da credencial da variável de ambiente configurada no arquivo .env
cred_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if cred_path:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = cred_path

# Inicializa o cliente do BigQuery
client = bigquery.Client()

print(f"Conexão com o BigQuery configurada com sucesso!")

Conexão com o BigQuery configurada com sucesso!


In [9]:
def ingest_from_bigquery(table_id: str, entity_name: str, force_offline: bool = False) -> pd.DataFrame:
    """
    Ingere dados de uma tabela do BigQuery e salva na camada Bronze com histórico.
    Se force_offline=True ou se houver falha na API, utiliza o fallback local 
    emitindo um alerta explícito de auditoria.
    """
    output_file = f"{BRONZE_PATH}/{entity_name}_raw_{INGESTION_TIMESTAMP}.parquet"
    
    # Modo offline forçado (para testes locais sem gastar cota/rede)
    if force_offline:
        print(f"{Fore.YELLOW} [MODO OFFLINE ATIVADO] Pulando BigQuery para [{entity_name}].")
        return _carregar_fallback_recente(entity_name)

    print(f"{Fore.CYAN} Iniciando ingestão da entidade [{entity_name}] do BigQuery...")
    print(f"    Tabela fonte: {table_id}")

    try:
        # 1. Executa a consulta no BigQuery
        query = f"SELECT * FROM `{table_id}`"
        query_job = client.query(query)

        # EVIDÊNCIA DE NUVEM: Exibe o ID do Job executado diretamente nos servidores do GCP
        print(f"[Cloud GCP] Job ID gerado na nuvem: {query_job.job_id}")

        df = client.query(query).to_dataframe(create_bqstorage_client=True)

        # 2. Metadados e Hash de rastreabilidade
        df['_ingestion_timestamp'] = INGESTION_TIMESTAMP
        df['_source_table']        = table_id
        df['_source_system']       = 'basedosdados_bigquery'
        
        df['_record_hash'] = df.drop(
            columns=['_ingestion_timestamp', '_source_table', '_source_system'],
            errors='ignore'
        ).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

        # 3. Salva com histórico na Bronze
        df.to_parquet(output_file, index=False)

        print(f"{Fore.GREEN}    {len(df)} registros ingeridos do BigQuery com sucesso.")
        print(f"    Salvo com histórico em: {output_file}")
        return df

    except Exception as e:
        # ALERTA VERMELHO/AMARELO CLARO: O erro não fica escondido, ele é logado explicitamente
        print(f"\n{Fore.RED}========================================================")
        print(f"{Fore.RED} [ALERTA DE FALHA] Erro crítico ao conectar no BigQuery para [{entity_name}]:")
        print(f"{Fore.YELLOW} Detalhe do erro: {e}")
        print(f"{Fore.RED}========================================================\n")
        
        print(f"{Fore.YELLOW} [CONTINGÊNCIA] O pipeline acionou o fallback para evitar parada total.")
        return _carregar_fallback_recente(entity_name)

def _carregar_fallback_recente(entity_name: str) -> pd.DataFrame:
    """Função auxiliar para buscar o arquivo mais recente na Bronze com aviso explícito."""
    search_pattern = f"{BRONZE_PATH}/{entity_name}_raw_*.parquet"
    arquivos_existentes = glob.glob(search_pattern)
    
    if arquivos_existentes:
        latest_file = max(arquivos_existentes)
        print(f"{Fore.MAGENTA}    [AUDITORIA] Carregando cache local pré-existente: {latest_file}")
        df = pd.read_parquet(latest_file)
        return df
    else:
        print(f"{Fore.RED}    [ERRO FATAL] Nenhum cache local encontrado para {entity_name}. Impossível prosseguir.")
        raise FileNotFoundError(f"Nenhum dado local disponível para {entity_name}")

print(f"{Fore.GREEN} Função de ingestão configurada")

 Função de ingestão configurada


In [10]:
# ============================================================
# ETAPA 2.2 — INGESTÃO MULTI-ENTIDADE
# ============================================================
# Ingerimos as entidades do projeto de avaliação da alfabetização:
# - dicionario                → dicionário de dados / variáveis
# - municipio                → indicadores consolidados por município
# - uf                       → indicadores consolidados por UF
# - meta_alfabetizacao_brasil  → trajetória e metas nacionais
# - meta_alfabetizacao_uf      → trajetória e metas por UF
# - meta_alfabetizacao_municipio → trajetória e metas por município
# - alunos

print(f"{Fore.YELLOW}{'='*60}")
print(f"{Fore.YELLOW}  INICIANDO PIPELINE DE INGESTÃO (BIGQUERY)")
print(f"{Fore.YELLOW}{'='*60}\n")

# Dicionário com todas as entidades a ingerir do BigQuery (Base dos Dados)
entities_bigquery = {
    'municipio':                    'basedosdados.br_inep_avaliacao_alfabetizacao.municipio',
    'uf':                           'basedosdados.br_inep_avaliacao_alfabetizacao.uf',
    'meta_alfabetizacao_brasil':    'basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil',
    'meta_alfabetizacao_uf':        'basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf',
    'meta_alfabetizacao_municipio': 'basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio',
    'dicionario':                   'basedosdados.br_inep_avaliacao_alfabetizacao.dicionario',
    'alunos':                       'basedosdados.br_inep_avaliacao_alfabetizacao.alunos'  
}

bronze_dfs = {}

for entity, table_id in entities_bigquery.items():
    bronze_dfs[entity] = ingest_from_bigquery(table_id, entity)
    print()

print(f"{Fore.GREEN}{'='*60}")
print(f"{Fore.GREEN}  INGESTÃO CONCLUÍDA")
print(f"{Fore.GREEN}   Total de entidades: {len(bronze_dfs)}")
print(f"{Fore.GREEN}   Total de registros: {sum(len(df) for df in bronze_dfs.values())}")
print(f"{Fore.GREEN}{'='*60}")

  INICIANDO PIPELINE DE INGESTÃO (BIGQUERY)

 Iniciando ingestão da entidade [municipio] do BigQuery...
    Tabela fonte: basedosdados.br_inep_avaliacao_alfabetizacao.municipio
[Cloud GCP] Job ID gerado na nuvem: a5bdab65-2016-46c3-94f3-a153f524da75
    23995 registros ingeridos do BigQuery com sucesso.
    Salvo com histórico em: ./data_lake/bronze/municipio_raw_20261031_212548.parquet

 Iniciando ingestão da entidade [uf] do BigQuery...
    Tabela fonte: basedosdados.br_inep_avaliacao_alfabetizacao.uf
[Cloud GCP] Job ID gerado na nuvem: 31d7cf4f-a135-4138-bfc9-e223960352c6
    145 registros ingeridos do BigQuery com sucesso.
    Salvo com histórico em: ./data_lake/bronze/uf_raw_20261031_212548.parquet

 Iniciando ingestão da entidade [meta_alfabetizacao_brasil] do BigQuery...
    Tabela fonte: basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
[Cloud GCP] Job ID gerado na nuvem: 813fa963-6930-40e4-a9a6-38353fe01b40
    3 registros ingeridos do BigQuery com sucesso

In [11]:
# ============================================================
# ETAPA 2.3 — INSPEÇÃO DOS DADOS BRUTOS
# ============================================================
# Sempre inspecionar os dados antes de processá-los!

print(f"{Fore.YELLOW} PRÉVIA DOS DADOS BRUTOS (Bronze)\n")

for entity, df in bronze_dfs.items():
    print(f"{Fore.CYAN}{'─'*50}")
    print(f"{Fore.CYAN}  Entidade: {entity.upper()}")
    print(f"   Shape (Linhas, Colunas): {df.shape} | Nulos totais: {df.isnull().sum().sum()}")
    print(f"   Tipos: { {col: str(dtype) for col, dtype in df.dtypes.items()} }")
    print()
    display(df.head(3))
    print()

 PRÉVIA DOS DADOS BRUTOS (Bronze)

──────────────────────────────────────────────────
  Entidade: MUNICIPIO
   Shape (Linhas, Colunas): (23995, 19) | Nulos totais: 103923
   Tipos: {'ano': 'Int64', 'id_municipio': 'object', 'serie': 'object', 'rede': 'object', 'taxa_alfabetizacao': 'float64', 'media_portugues': 'float64', 'proporcao_aluno_nivel_0': 'float64', 'proporcao_aluno_nivel_1': 'float64', 'proporcao_aluno_nivel_2': 'float64', 'proporcao_aluno_nivel_3': 'float64', 'proporcao_aluno_nivel_4': 'float64', 'proporcao_aluno_nivel_5': 'float64', 'proporcao_aluno_nivel_6': 'float64', 'proporcao_aluno_nivel_7': 'float64', 'proporcao_aluno_nivel_8': 'float64', '_ingestion_timestamp': 'object', '_source_table': 'object', '_source_system': 'object', '_record_hash': 'object'}



,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2023,1100031,2,3,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,d1b6df4e80959ce1a5082a62c2f1db0c
1,2023,1100072,2,3,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,4cda7a8d76f3afa7a02c2237a0868043
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,5b53ac3da3f4a8d4c8a16760b6a46b74



──────────────────────────────────────────────────
  Entidade: UF
   Shape (Linhas, Colunas): (145, 19) | Nulos totais: 630
   Tipos: {'ano': 'Int64', 'sigla_uf': 'object', 'serie': 'object', 'rede': 'object', 'taxa_alfabetizacao': 'float64', 'media_portugues': 'float64', 'proporcao_aluno_nivel_0': 'float64', 'proporcao_aluno_nivel_1': 'float64', 'proporcao_aluno_nivel_2': 'float64', 'proporcao_aluno_nivel_3': 'float64', 'proporcao_aluno_nivel_4': 'float64', 'proporcao_aluno_nivel_5': 'float64', 'proporcao_aluno_nivel_6': 'float64', 'proporcao_aluno_nivel_7': 'float64', 'proporcao_aluno_nivel_8': 'float64', '_ingestion_timestamp': 'object', '_source_table': 'object', '_source_system': 'object', '_record_hash': 'object'}



,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.uf,basedosdados_bigquery,42ee6a943b5c1b9714c8e978c57f248b
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.uf,basedosdados_bigquery,eca51cb864fb5c8429b4b562aa79eee6
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.uf,basedosdados_bigquery,586978b39f93e4d089e198ec3945236a



──────────────────────────────────────────────────
  Entidade: META_ALFABETIZACAO_BRASIL
   Shape (Linhas, Colunas): (3, 15) | Nulos totais: 0
   Tipos: {'ano': 'Int64', 'rede': 'object', 'taxa_alfabetizacao': 'float64', 'meta_alfabetizacao_2024': 'float64', 'meta_alfabetizacao_2025': 'float64', 'meta_alfabetizacao_2026': 'float64', 'meta_alfabetizacao_2027': 'float64', 'meta_alfabetizacao_2028': 'float64', 'meta_alfabetizacao_2029': 'float64', 'meta_alfabetizacao_2030': 'float64', 'percentual_participacao': 'float64', '_ingestion_timestamp': 'object', '_source_table': 'object', '_source_system': 'object', '_record_hash': 'object'}



,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2025,Pública,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0,88.00,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,fc91a9077697cf8711b674e9889d2cab
1,2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0,87.37,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,14eac9e1c9a7e2bd9221c6cdd7647110
2,2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0,86.00,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,5730997b67cf1c69fa2b61018ffe34c0


... (4 outputs left out)


──────────────────────────────────────────────────
  Entidade: DICIONARIO
   Shape (Linhas, Colunas): (27, 9) | Nulos totais: 0
   Tipos: {'id_tabela': 'object', 'nome_coluna': 'object', 'chave': 'object', 'cobertura_temporal': 'object', 'valor': 'object', '_ingestion_timestamp': 'object', '_source_table': 'object', '_source_system': 'object', '_record_hash': 'object'}



,id_tabela,nome_coluna,chave,cobertura_temporal,valor,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,alunos,alfabetizado,0,(1),Não,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.d...,basedosdados_bigquery,ab2fea563eb9aaf844ce5653f4ee7715
1,alunos,alfabetizado,1,(1),Sim,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.d...,basedosdados_bigquery,ffafb21c590d647d55c00c816f5d0a43
2,alunos,preenchimento_caderno,0,(1),Prova não preenchida,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.d...,basedosdados_bigquery,9defe376b673a1f661852a24a066ea2b



──────────────────────────────────────────────────
  Entidade: ALUNOS
   Shape (Linhas, Colunas): (3867999, 16) | Nulos totais: 1026676
   Tipos: {'ano': 'Int64', 'id_municipio': 'object', 'id_escola': 'object', 'id_aluno': 'object', 'caderno': 'object', 'serie': 'object', 'rede': 'object', 'presenca': 'object', 'preenchimento_caderno': 'object', 'alfabetizado': 'object', 'proficiencia': 'float64', 'peso_aluno': 'float64', '_ingestion_timestamp': 'object', '_source_table': 'object', '_source_system': 'object', '_record_hash': 'object'}



,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2023,1302702,60000861,13050049,1,2,2,0,0,0,NaN,NaN,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.a...,basedosdados_bigquery,3f39a648a9a618bb3c6a167c5be2840f
1,2023,1502301,60003317,15036836,1,2,3,0,0,0,NaN,NaN,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.a...,basedosdados_bigquery,c46adff1e7ddaaafa04e6cf8cb84c743
2,2023,2925600,60015387,29002672,1,2,3,0,0,0,NaN,NaN,20261031_212548,basedosdados.br_inep_avaliacao_alfabetizacao.a...,basedosdados_bigquery,aaf7429a9908043882f195e8256841e2


# Etapa 3 - Camada SILVER: Limpeza e Normalização

Nesta etapa, os dados brutos da camada Bronze passam pelo processo de transformação, higienização e padronização. O objetivo da camada Silver é garantir a qualidade e a consistência técnica das bases antes do consumo analítico.

### Diretrizes e Regras de Transformação:

* **Padronização de Tipos e Chaves:** Ajuste de schemas, normalização de nomes de colunas e conversão explícita de tipos de dados para permitir a integração precisa entre diferentes tabelas.
* **Tratamento de Valores Ausentes:** Identificação e tratamento de valores nulos respeitando as regras de negócio, preservando nulos quando necessário para evitar distorções em estatísticas e escalas de proficiência.
* **Desduplicação e Qualidade:** Remoção de registros duplicados e validação de regras funcionais antes de disponibilizar as tabelas tratadas.
* **Cruzamento com Dicionário Oficial:** Mapeamento de códigos categóricos utilizando a tabela de dicionário original para padronizar descrições de rede, série e presença.
* **Persistência em Parquet:** Armazenamento dos datasets limpos e otimizados no diretório `./data_lake/silver` em formato Parquet.

---
**Saídas em `data_lake/silver/`:** Tabelas tratadas de alunos, municípios, UFs, trajetórias de metas e dicionário prontas para a consolidação dos indicadores.

In [12]:
# ============================================================
# ETAPA 3.1 — TRANSFORMAÇÃO: MUNICIPIO
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: MUNICIPIO\n")

# 1. Cópia do dataframe bronze
df_municipio_silver = bronze_dfs['municipio'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_municipio_silver)
df_municipio_silver = df_municipio_silver.drop_duplicates()
linhas_depois = len(df_municipio_silver)

print(f"Duplicatas removidas: {linhas_antes - linhas_depois}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_municipio_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_municipio_silver['id_municipio'] = df_municipio_silver['id_municipio'].astype(str)
df_municipio_silver['serie'] = df_municipio_silver['serie'].astype(str)
df_municipio_silver['rede'] = df_municipio_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao', 'media_portugues'] + [col for col in df_municipio_silver.columns if 'proporcao_aluno_nivel' in col]
for col in colunas_numericas:
    df_municipio_silver[col] = pd.to_numeric(df_municipio_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_municipio_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_municipio_silver['_record_hash'] = df_municipio_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/municipio_silver.parquet"
df_municipio_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade MUNICIPIO processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_municipio_silver)}")

  Processando entidade: MUNICIPIO



Duplicatas removidas: 0

Valores nulos por coluna:
ano                            0
id_municipio                   0
serie                          0
rede                           0
taxa_alfabetizacao             0
media_portugues                0
proporcao_aluno_nivel_0    11547
proporcao_aluno_nivel_1    11547
proporcao_aluno_nivel_2    11547
proporcao_aluno_nivel_3    11547
proporcao_aluno_nivel_4    11547
proporcao_aluno_nivel_5    11547
proporcao_aluno_nivel_6    11547
proporcao_aluno_nivel_7    11547
proporcao_aluno_nivel_8    11547
_ingestion_timestamp           0
_source_table                  0
_source_system                 0
_record_hash                   0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
id_municipio                object
serie                       object
rede                        object
taxa_alfabetizacao         float64
media_portugues            float64
proporcao_aluno_nivel_0    float64
proporcao_aluno_nivel_1    float64


In [13]:
# ============================================================
# ETAPA 3.2 — TRANSFORMAÇÃO: UF
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: UF\n")

# 1. Cópia do dataframe bronze
df_uf_silver = bronze_dfs['uf'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_uf_silver)
df_uf_silver = df_uf_silver.drop_duplicates()
linhas_depois = len(df_uf_silver)

print(f"Duplicatas removidas: {linhas_antes - linhas_depois}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_uf_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_uf_silver['sigla_uf'] = df_uf_silver['sigla_uf'].astype(str)
df_uf_silver['serie'] = df_uf_silver['serie'].astype(str)
df_uf_silver['rede'] = df_uf_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao', 'media_portugues'] + [col for col in df_uf_silver.columns if 'proporcao_aluno_nivel' in col]
for col in colunas_numericas:
    df_uf_silver[col] = pd.to_numeric(df_uf_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_uf_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_uf_silver['_record_hash'] = df_uf_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/uf_silver.parquet"
df_uf_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade UF processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_uf_silver)}")

  Processando entidade: UF

Duplicatas removidas: 0

Valores nulos por coluna:
ano                         0
sigla_uf                    0
serie                       0
rede                        0
taxa_alfabetizacao          0
media_portugues             0
proporcao_aluno_nivel_0    70
proporcao_aluno_nivel_1    70
proporcao_aluno_nivel_2    70
proporcao_aluno_nivel_3    70
proporcao_aluno_nivel_4    70
proporcao_aluno_nivel_5    70
proporcao_aluno_nivel_6    70
proporcao_aluno_nivel_7    70
proporcao_aluno_nivel_8    70
_ingestion_timestamp        0
_source_table               0
_source_system              0
_record_hash                0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
sigla_uf                    object
serie                       object
rede                        object
taxa_alfabetizacao         float64
media_portugues            float64
proporcao_aluno_nivel_0    float64
proporcao_aluno_nivel_1    float64
proporcao_aluno_nivel_2    fl

In [14]:
# ============================================================
# ETAPA 3.3 — TRANSFORMAÇÃO: META_ALFABETIZACAO_BRASIL
# ============================================================
print(f"{Fore.YELLOW} Processando entidade: META_ALFABETIZACAO_BRASIL\n")

# 1. Cópia do dataframe bronze
df_meta_alfabetizacao_brasil_silver = bronze_dfs['meta_alfabetizacao_brasil'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_meta_alfabetizacao_brasil_silver)
df_meta_alfabetizacao_brasil_silver = df_meta_alfabetizacao_brasil_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_meta_alfabetizacao_brasil_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_meta_alfabetizacao_brasil_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_meta_alfabetizacao_brasil_silver['rede'] = df_meta_alfabetizacao_brasil_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao'] + [col for col in df_meta_alfabetizacao_brasil_silver.columns if 'meta_alfabetizacao_' in col]
for col in colunas_numericas:
    df_meta_alfabetizacao_brasil_silver[col] = pd.to_numeric(df_meta_alfabetizacao_brasil_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_meta_alfabetizacao_brasil_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_meta_alfabetizacao_brasil_silver['_record_hash'] = df_meta_alfabetizacao_brasil_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/meta_alfabetizacao_brasil_silver.parquet"
df_meta_alfabetizacao_brasil_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade META_ALFABETIZACAO_BRASIL processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_meta_alfabetizacao_brasil_silver)}")

 Processando entidade: META_ALFABETIZACAO_BRASIL

Duplicatas removidas: 0

Valores nulos por coluna:
ano                        0
rede                       0
taxa_alfabetizacao         0
meta_alfabetizacao_2024    0
meta_alfabetizacao_2025    0
meta_alfabetizacao_2026    0
meta_alfabetizacao_2027    0
meta_alfabetizacao_2028    0
meta_alfabetizacao_2029    0
meta_alfabetizacao_2030    0
percentual_participacao    0
_ingestion_timestamp       0
_source_table              0
_source_system             0
_record_hash               0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
rede                        object
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabetizacao_2029    float64
meta_alfabetizacao_2030    float64
percentual_participacao    float64
_ingestion_timestamp        object
_s

In [15]:
# ============================================================
# ETAPA 3.4 — TRANSFORMAÇÃO: META_ALFABETIZACAO_UF
# ============================================================
print(f"{Fore.YELLOW} Processando entidade: META_ALFABETIZACAO_UF\n")

# 1. Cópia do dataframe bronze
df_meta_alfabetizacao_uf_silver = bronze_dfs['meta_alfabetizacao_uf'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_meta_alfabetizacao_uf_silver)
df_meta_alfabetizacao_uf_silver = df_meta_alfabetizacao_uf_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_meta_alfabetizacao_uf_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_meta_alfabetizacao_uf_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_meta_alfabetizacao_uf_silver['sigla_uf'] = df_meta_alfabetizacao_uf_silver['sigla_uf'].astype(str)
df_meta_alfabetizacao_uf_silver['rede'] = df_meta_alfabetizacao_uf_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao'] + [col for col in df_meta_alfabetizacao_uf_silver.columns if 'meta_alfabetizacao_' in col]
for col in colunas_numericas:
    df_meta_alfabetizacao_uf_silver[col] = pd.to_numeric(df_meta_alfabetizacao_uf_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_meta_alfabetizacao_uf_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_meta_alfabetizacao_uf_silver['_record_hash'] = df_meta_alfabetizacao_uf_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/meta_alfabetizacao_uf_silver.parquet"
df_meta_alfabetizacao_uf_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade META_ALFABETIZACAO_UF processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_meta_alfabetizacao_uf_silver)}")

 Processando entidade: META_ALFABETIZACAO_UF

Duplicatas removidas: 0

Valores nulos por coluna:
ano                        0
sigla_uf                   0
rede                       0
taxa_alfabetizacao         4
meta_alfabetizacao_2024    9
meta_alfabetizacao_2025    3
meta_alfabetizacao_2026    2
meta_alfabetizacao_2027    2
meta_alfabetizacao_2028    2
meta_alfabetizacao_2029    2
meta_alfabetizacao_2030    2
percentual_participacao    4
_ingestion_timestamp       0
_source_table              0
_source_system             0
_record_hash               0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
sigla_uf                    object
rede                        object
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabetizacao_2029    float64
meta_alfabetizacao_2030    float64
percentual_p

In [16]:
# ============================================================
# ETAPA 3.5 — TRANSFORMAÇÃO: META_ALFABETIZACAO_MUNICIPIO
# ============================================================
print(f"{Fore.YELLOW} Processando entidade: META_ALFABETIZACAO_MUNICIPIO\n")

# 1. Cópia do dataframe bronze
df_meta_alfabetizacao_municipio_silver = bronze_dfs['meta_alfabetizacao_municipio'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_meta_alfabetizacao_municipio_silver)
df_meta_alfabetizacao_municipio_silver = df_meta_alfabetizacao_municipio_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_meta_alfabetizacao_municipio_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_meta_alfabetizacao_municipio_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_meta_alfabetizacao_municipio_silver['id_municipio'] = df_meta_alfabetizacao_municipio_silver['id_municipio'].astype(str)
df_meta_alfabetizacao_municipio_silver['rede'] = df_meta_alfabetizacao_municipio_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao'] + [col for col in df_meta_alfabetizacao_municipio_silver.columns if 'meta_alfabetizacao_' in col]
for col in colunas_numericas:
    df_meta_alfabetizacao_municipio_silver[col] = pd.to_numeric(df_meta_alfabetizacao_municipio_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_meta_alfabetizacao_municipio_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_meta_alfabetizacao_municipio_silver['_record_hash'] = df_meta_alfabetizacao_municipio_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/meta_alfabetizacao_municipio_silver.parquet"
df_meta_alfabetizacao_municipio_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade META_ALFABETIZACAO_MUNICIPIO processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_meta_alfabetizacao_municipio_silver)}")

 Processando entidade: META_ALFABETIZACAO_MUNICIPIO

Duplicatas removidas: 0

Valores nulos por coluna:
ano                          0
id_municipio                 0
rede                         0
taxa_alfabetizacao         120
meta_alfabetizacao_2024    240
meta_alfabetizacao_2025      0
meta_alfabetizacao_2026      0
meta_alfabetizacao_2027      0
meta_alfabetizacao_2028      0
meta_alfabetizacao_2029      0
meta_alfabetizacao_2030      0
nivel_alfabetizacao        120
percentual_participacao    120
_ingestion_timestamp         0
_source_table                0
_source_system               0
_record_hash                 0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
id_municipio                object
rede                        object
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabet

In [17]:
# ============================================================
# ETAPA 3.6 — TRANSFORMAÇÃO: DICIONARIO
# ============================================================
print(f"{Fore.YELLOW} Processando entidade: DICIONARIO\n")

# 1. Cópia do dataframe bronze
df_dicionario_silver = bronze_dfs['dicionario'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_dicionario_silver)
df_dicionario_silver = df_dicionario_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_dicionario_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_dicionario_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
colunas_texto = ['id_tabela', 'nome_coluna', 'chave', 'cobertura_temporal', 'valor']
for col in colunas_texto:
    if col in df_dicionario_silver.columns:
        df_dicionario_silver[col] = df_dicionario_silver[col].astype(str)

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_dicionario_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_dicionario_silver['_record_hash'] = df_dicionario_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/dicionario_silver.parquet"
df_dicionario_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade DICIONARIO processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_dicionario_silver)}")

 Processando entidade: DICIONARIO

Duplicatas removidas: 0

Valores nulos por coluna:
id_tabela               0
nome_coluna             0
chave                   0
cobertura_temporal      0
valor                   0
_ingestion_timestamp    0
_source_table           0
_source_system          0
_record_hash            0
dtype: int64

Tipos de dados atualizados:
id_tabela               object
nome_coluna             object
chave                   object
cobertura_temporal      object
valor                   object
_ingestion_timestamp    object
_source_table           object
_source_system          object
_record_hash            object
dtype: object
 Entidade DICIONARIO processada com sucesso na camada Silver!
 Salvo em: ./data_lake/silver/dicionario_silver.parquet
 Total de registros: 27


In [18]:
# ============================================================
# ETAPA 3.7 — TRANSFORMAÇÃO: ALUNOS
# ============================================================
print(f"{Fore.YELLOW} Processando entidade: ALUNOS\n")

# 1. Cópia do dataframe bronze
df_alunos_silver = bronze_dfs['alunos'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_alunos_silver)
df_alunos_silver = df_alunos_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_alunos_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_alunos_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
colunas_texto = ['id_municipio', 'id_escola', 'id_aluno', 'caderno', 'serie', 'rede', 'presenca', 'preenchimento_caderno', 'alfabetizado']
for col in colunas_texto:
    if col in df_alunos_silver.columns:
        df_alunos_silver[col] = df_alunos_silver[col].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['proficiencia', 'peso_aluno']
for col in colunas_numericas:
    if col in df_alunos_silver.columns:
        df_alunos_silver[col] = pd.to_numeric(df_alunos_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_alunos_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_alunos_silver['_record_hash'] = df_alunos_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/alunos_silver.parquet"
df_alunos_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade ALUNOS processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_alunos_silver)}")

 Processando entidade: ALUNOS

Duplicatas removidas: 0

Valores nulos por coluna:
ano                           0
id_municipio                  0
id_escola                     0
id_aluno                      0
caderno                       0
serie                         0
rede                          0
presenca                      0
preenchimento_caderno         0
alfabetizado                  0
proficiencia             513338
peso_aluno               513338
_ingestion_timestamp          0
_source_table                 0
_source_system                0
_record_hash                  0
dtype: int64

Tipos de dados atualizados:
ano                        Int64
id_municipio              object
id_escola                 object
id_aluno                  object
caderno                   object
serie                     object
rede                      object
presenca                  object
preenchimento_caderno     object
alfabetizado              object
proficiencia             float64
p

# Etapa 4 - Camada GOLD: Indicadores e Análises

Aqui a Silver vira os indicadores finais: resultado vs meta (município, UF e Brasil), evolução no tempo, validação com os microdados e a base para a Fase 3.

Alguns cuidados que guiaram as tabelas:

- resultado e meta só são comparados na mesma rede e na série 2 (as metas são do 2º ano)
- quem não tem meta fica como `sem meta cadastrada`, e não como "não atingiu"
- médias entre localidades são ponderadas pelos alunos avaliados
- a distância até 2030 usa a meta pactuada da fonte (80 na rede pública); os 100% aparecem só como cenário aspiracional
- a projeção até 2030 é um cenário de tendência, não previsão
- nulo continua nulo
- a coluna `origem` separa dado oficial de simulado (streaming)
- todo join tem a taxa de correspondência medida e gravada em `quality_reports/gold_checks.csv`; se ficar abaixo do limiar, o processo para

Saídas em `data_lake/gold/`: `gold_municipio_resultado_vs_meta`, `gold_uf_resultado_vs_meta`, `gold_brasil_resultado_vs_meta`, `gold_validacao_microdado_vs_oficial`, `gold_evolucao_indicador`, `gold_cenario_2030`, `gold_features_fase3` e `gold_numeros_video.csv`.

In [19]:
# --- 4.0 setup da gold
import unicodedata

GOLD_RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

PONTO_CORTE_SAEB = 743
SERIE_ALVO = '2'                    # metas são do 2º ano
CENARIO_ASPIRACIONAL_2030 = 100.0   # só cenário; distância é sempre contra a meta pactuada da fonte
TOLERANCIA_PROVA_DOS_NOVE_PP = 0.5

# --- mínimo de linhas com par em cada join
# municipio x meta: quase todos aderiram, abaixo de 90% é chave errada
# uf x meta: 95% = no máximo 1 UF sem meta
# microdados/agregados: só informativo
LIMIAR_JOIN = {
    'municipio_x_meta_municipio': 0.90,
    'uf_x_meta_uf':               0.95,
    'municipio_x_pesos_alunos':   0.95,
    'uf_x_pesos_alunos':          0.95,
    'brasil_x_agregado_ufs':      1.00,
    'brasil_x_microdado':         1.00,
    'uf_x_microdado':             0.95,
}

UF_POR_CODIGO_IBGE = {
    '11': 'RO', '12': 'AC', '13': 'AM', '14': 'RR', '15': 'PA', '16': 'AP', '17': 'TO',
    '21': 'MA', '22': 'PI', '23': 'CE', '24': 'RN', '25': 'PB', '26': 'PE', '27': 'AL',
    '28': 'SE', '29': 'BA', '31': 'MG', '32': 'ES', '33': 'RJ', '35': 'SP', '41': 'PR',
    '42': 'SC', '43': 'RS', '50': 'MS', '51': 'MT', '52': 'GO', '53': 'DF',
}

# --- redes agregadas do dicionário (0, 5, 6) -> redes simples que somam
REDES_AGREGADAS = {
    'publica_estadual_municipal':         ['estadual', 'municipal'],
    'publica_federal_estadual_municipal': ['federal', 'estadual', 'municipal'],
    'total':                              ['federal', 'estadual', 'municipal', 'privada'],
}

# --- meta diz só "Pública"; pelos dados é a rede 5 (bate em 49 de 49 UFs)
REDE_PUBLICA_DAS_METAS = 'publica_estadual_municipal'

ARQUIVO_CHECKS_GOLD = f"{QUALITY_PATH}/gold_checks.csv"
ARQUIVO_ANOMALIAS = f"{QUALITY_PATH}/gold_anomalias.csv"


class FalhaQualidadeBloqueante(Exception):
    pass


def registrar_check(regra, tabela, tipo, aprovado, valor=None, limiar=None, detalhe=''):
    # --- grava em csv (append) e para tudo se a bloqueante falhar
    aprovado = bool(aprovado)
    registro = pd.DataFrame([{
        'gold_run_ts':   GOLD_RUN_TS,
        'regra':         regra,
        'tabela':        tabela,
        'tipo':          tipo,
        'status':        'APROVADO' if aprovado else 'REPROVADO',
        'valor':         valor,
        'limiar':        limiar,
        'detalhe':       detalhe,
        'registrado_em': datetime.now(timezone.utc).isoformat(),
    }])
    registro.to_csv(ARQUIVO_CHECKS_GOLD, mode='a', index=False,
                    header=not os.path.exists(ARQUIVO_CHECKS_GOLD))

    cor = Fore.GREEN if aprovado else (Fore.RED if tipo == 'bloqueante' else Fore.YELLOW)
    print(f"{cor}   [{tipo.upper()}] {regra} ({tabela}): {'OK' if aprovado else 'FALHOU'}"
          f" | valor={valor} limiar={limiar} {detalhe}")
    if not aprovado and tipo == 'bloqueante':
        raise FalhaQualidadeBloqueante(f"{regra} em {tabela}: valor={valor}, limiar={limiar}. {detalhe}")


def checar_chave_unica(df, chaves, tabela):
    n_dup = int(df.duplicated(subset=chaves).sum())
    registrar_check(f"chave_unica:{'+'.join(chaves)}", tabela, 'bloqueante', n_dup == 0, n_dup, 0)


def join_medido(esquerda, direita, chaves, nome_join, tabela, bloqueante=True):
    # --- left join + tipo das chaves + % de linhas com par
    tipos_diferentes = [f"{k}: {esquerda[k].dtype} x {direita[k].dtype}"
                        for k in chaves if str(esquerda[k].dtype) != str(direita[k].dtype)]
    registrar_check(f"tipos_chaves_compativeis:{nome_join}", tabela, 'bloqueante',
                    not tipos_diferentes, len(tipos_diferentes), 0, '; '.join(tipos_diferentes))

    df = esquerda.merge(direita, on=chaves, how='left', indicator='_match', validate='many_to_one')
    n_par = int((df['_match'] == 'both').sum())
    taxa = n_par / len(df) if len(df) else 0.0
    limiar = LIMIAR_JOIN[nome_join]
    registrar_check(f"taxa_correspondencia_join:{nome_join}", tabela,
                    'bloqueante' if bloqueante else 'informativa',
                    taxa >= limiar, round(taxa, 4), limiar, f"{n_par}/{len(df)} linhas com par")
    return df.drop(columns='_match')


def normalizar_texto(valor):
    if pd.isna(valor):
        return valor
    return unicodedata.normalize('NFKD', str(valor)).encode('ascii', 'ignore').decode().lower().strip()


def mapa_dicionario(tabela, coluna):
    d = df_dicionario_gold[(df_dicionario_gold['id_tabela'] == tabela) &
                           (df_dicionario_gold['nome_coluna'] == coluna)]
    return dict(zip(d['chave'], d['valor']))


def traduzir_codigo(serie_codigos, tabela, coluna, tabela_gold):
    # --- código fora do dicionário vira 'Desconhecido' e reprova
    mapa = mapa_dicionario(tabela, coluna)
    if not mapa:
        print(f"{Fore.YELLOW}   Dicionário não traduz {tabela}.{coluna}; códigos mantidos como estão.")
        return None
    descricao = serie_codigos.map(mapa).fillna('Desconhecido')
    desconhecidos = sorted(serie_codigos[descricao == 'Desconhecido'].dropna().unique().tolist())
    registrar_check(f"codigo_desconhecido:{tabela}.{coluna}", tabela_gold, 'bloqueante',
                    len(desconhecidos) == 0, int((descricao == 'Desconhecido').sum()), 0,
                    f"códigos fora do dicionário: {desconhecidos[:10]}" if desconhecidos else '')
    return descricao


def chave_rede(descricao):
    # --- 'Pública (Estadual e Municipal)' -> 'publica_estadual_municipal', 'Municipal' -> 'municipal'
    simples = ['privada', 'federal', 'estadual', 'municipal']
    t = normalizar_texto(descricao)
    if pd.isna(t):
        return 'desconhecido'
    if t in simples:
        return t
    if t == 'publica':
        return REDE_PUBLICA_DAS_METAS
    if t.startswith('total'):
        return 'total'
    if t.startswith('publica'):
        partes = [r for r in ['federal', 'estadual', 'municipal'] if r in t]
        chave = '_'.join(['publica'] + partes)
        return chave if chave in REDES_AGREGADAS else 'desconhecido'
    return 'desconhecido'


def adicionar_chave_rede(df, coluna_descricao, tabela_gold):
    df['rede_chave'] = df[coluna_descricao].map(chave_rede).astype('str')
    n_desc = int((df['rede_chave'] == 'desconhecido').sum())
    registrar_check('rede_desconhecida', tabela_gold, 'bloqueante', n_desc == 0, n_desc, 0,
                    f"descrições sem chave: {df.loc[df['rede_chave'] == 'desconhecido', coluna_descricao].unique()[:5].tolist()}"
                    if n_desc else '')
    return df


def checar_rede_injetiva(tabela_dic, tabela_gold):
    # --- dois códigos não podem virar a mesma chave
    mapa = mapa_dicionario(tabela_dic, 'rede')
    tabela_mapa = pd.DataFrame({'codigo': list(mapa.keys()), 'descricao': list(mapa.values())})
    tabela_mapa['rede_chave'] = tabela_mapa['descricao'].map(chave_rede)
    print(f"{Fore.CYAN}   Mapeamento de rede ({tabela_dic}):")
    print(tabulate(tabela_mapa, headers='keys', tablefmt='simple', showindex=False))
    colisoes = tabela_mapa[tabela_mapa['rede_chave'].duplicated(keep=False)]
    registrar_check(f"rede_chave_injetiva:{tabela_dic}", tabela_gold, 'bloqueante',
                    colisoes.empty, len(colisoes), 0,
                    f"códigos colidindo: {colisoes['codigo'].tolist()}" if not colisoes.empty else '')


def extrair_meta_do_ano(df):
    # --- meta do ano Y vem da linha de ano Y (a trajetória é revisada); 2023 não tem meta
    meta = pd.Series(np.nan, index=df.index, dtype='float64')
    for ano in df['ano'].dropna().unique():
        coluna = f"meta_alfabetizacao_{int(ano)}"
        if coluna in df.columns:
            mascara = (df['ano'] == ano).fillna(False).to_numpy()
            meta[mascara] = df.loc[mascara, coluna].astype('float64')
    return meta


def classificar_meta(df, tem_linha_meta, ano_coberto):
    # --- sem meta != não atingiu
    taxa, meta = df['taxa_alfabetizacao'], df['meta_do_ano']
    df['status_meta'] = np.select(
        [meta.isna(), taxa.isna(), taxa >= meta],
        ['sem meta cadastrada', 'sem resultado', 'atingiu'],
        default='não atingiu',
    )
    primeiro_ano_meta = 2024
    df['motivo_sem_meta'] = np.select(
        [meta.notna(),
         ~ano_coberto,
         df['ano'].astype('float64').to_numpy() < primeiro_ano_meta,
         ~tem_linha_meta],
        [None,
         'ano sem tabela de meta publicada',
         'ano de linha de base (trajetória começa em 2024)',
         'localidade sem linha na tabela de meta'],
        default='meta do ano nula na fonte',
    )
    df['gap_meta_do_ano_pp'] = taxa - meta
    df['distancia_meta_pactuada_2030_pp'] = df['meta_alfabetizacao_2030'] - taxa
    df['distancia_cenario_aspiracional_2030_pp'] = CENARIO_ASPIRACIONAL_2030 - taxa
    return df


COLUNAS_META = [f"meta_alfabetizacao_{a}" for a in range(2024, 2031)]


def montar_resultado_vs_meta(resultado, meta, chaves_localidade, nome_join, tabela_gold, bloqueante):
    # --- left join partindo do resultado, pra quem não tem meta não sumir
    chaves = ['ano'] + chaves_localidade + ['rede_chave']
    meta = meta.assign(_tem_linha_meta=True)

    # --- ano sem tabela de meta não entra na medição do join
    anos_meta = set(meta['ano'].dropna().astype(int))
    coberto = resultado['ano'].astype(int).isin(anos_meta)
    anos_fora = sorted(set(resultado.loc[~coberto, 'ano'].astype(int)))
    registrar_check(f"anos_resultado_sem_tabela_meta:{nome_join}", tabela_gold, 'informativa',
                    not anos_fora, int((~coberto).sum()), 0, f"anos: {anos_fora}" if anos_fora else '')

    df = pd.concat([
        join_medido(resultado[coberto], meta, chaves, nome_join, tabela_gold, bloqueante=bloqueante),
        resultado[~coberto].merge(meta, on=chaves, how='left'),
    ], ignore_index=True)
    tem_linha_meta = df['_tem_linha_meta'].eq(True).to_numpy()
    ano_coberto = df['ano'].astype(int).isin(anos_meta).to_numpy()
    df['meta_do_ano'] = extrair_meta_do_ano(df)
    df = classificar_meta(df, tem_linha_meta, ano_coberto)
    return df.drop(columns='_tem_linha_meta')


def checar_consistencia_taxa(df, nome, tabela_gold):
    # --- taxa da tabela de resultado x taxa que vem na tabela de meta (se não bate, rede errada)
    comparaveis = df['taxa_alfabetizacao_tabela_meta'].notna() & df['taxa_alfabetizacao'].notna()
    diferenca = (df['taxa_alfabetizacao'] - df['taxa_alfabetizacao_tabela_meta']).abs()[comparaveis]
    proporcao = float((diferenca <= 0.5).mean()) if len(diferenca) else 1.0
    registrar_check(f"consistencia_taxa_{nome}_vs_tabela_meta", tabela_gold, 'bloqueante',
                    proporcao >= 0.95, round(proporcao, 4), 0.95,
                    f"{int((diferenca <= 0.01).sum())} idênticas e {int((diferenca <= 0.5).sum())} até 0,5 p.p. "
                    f"de {len(diferenca)} comparáveis; maior diferença {diferenca.max():.2f} p.p."
                    if len(diferenca) else 'sem linhas comparáveis')


def taxa_ponderada_microdado(df, chaves):
    # --- peso dos alfabetizados / peso dos avaliados
    agregado = (df.assign(_peso_alf=df['peso_aluno'] * df['_alfabetizado'])
                  .groupby(chaves, observed=True)
                  .agg(n_alunos_avaliados=('peso_aluno', 'size'),
                       soma_peso_avaliados=('peso_aluno', 'sum'),
                       soma_peso_alfabetizados=('_peso_alf', 'sum'))
                  .reset_index())
    agregado['taxa_microdado'] = 100 * agregado['soma_peso_alfabetizados'] / agregado['soma_peso_avaliados']
    return agregado


def com_redes_agregadas(df, chaves_sem_rede):
    # --- microdado só tem rede simples; monta pública/total somando
    partes = [df]
    for rede_agregada, componentes in REDES_AGREGADAS.items():
        agregado = (df[df['rede_chave'].isin(componentes)]
                      .groupby(chaves_sem_rede, observed=True)
                      .agg(n_alunos_avaliados=('n_alunos_avaliados', 'sum'),
                           soma_peso_avaliados=('soma_peso_avaliados', 'sum'),
                           soma_peso_alfabetizados=('soma_peso_alfabetizados', 'sum'))
                      .reset_index())
        agregado['rede_chave'] = rede_agregada
        agregado['taxa_microdado'] = 100 * agregado['soma_peso_alfabetizados'] / agregado['soma_peso_avaliados']
        partes.append(agregado)
    return pd.concat(partes, ignore_index=True)


def salvar_gold(df, nome):
    df = df.copy()
    df['_gold_run_ts'] = GOLD_RUN_TS
    caminho = f"{GOLD_PATH}/{nome}.parquet"
    df.to_parquet(caminho, index=False)
    gold_dfs[nome] = df
    print(f"{Fore.GREEN}   Salvo em: {caminho} ({len(df)} linhas, {df.shape[1]} colunas)")
    return df


gold_dfs = {}
df_dicionario_gold = pd.read_parquet(f"{SILVER_PATH}/dicionario_silver.parquet")

print(f"{Fore.GREEN} Setup da camada Gold concluído (execução {GOLD_RUN_TS}).")
print(f"   Checks de qualidade em: {ARQUIVO_CHECKS_GOLD}")

 Setup da camada Gold concluído (execução 20261031_212958).
   Checks de qualidade em: ./data_lake/quality_reports/gold_checks.csv


In [20]:
# ============================================================
# ETAPA 4.3 — ENRIQUECIMENTO DINÂMICO DA TABELA ALUNOS
# ============================================================
print(f"{Fore.YELLOW} Processando consolidação Gold: Alunos\n")

# 1. Carregar os dados limpos da Silver
df_alunos_silver = pd.read_parquet(f"{SILVER_PATH}/alunos_silver.parquet")

# 2. Descobrir dinamicamente quais colunas da tabela 'alunos' possuem mapeamento no dicionário
colunas_para_mapear = df_dicionario_silver[df_dicionario_silver['id_tabela'] == 'alunos']['nome_coluna'].unique()

print(f"Colunas identificadas dinamicamente para tradução: {list(colunas_para_mapear)}")

# 3. Função de mapeamento genérica e dinâmica
df_gold_alunos = df_alunos_silver.copy()

for coluna in colunas_para_mapear:
    # Extrai o dicionário (chave -> valor) para a coluna atual
    mapa = df_dicionario_silver[
        (df_dicionario_silver['id_tabela'] == 'alunos') & 
        (df_dicionario_silver['nome_coluna'] == coluna)
    ][['chave', 'valor']].set_index('chave')['valor'].to_dict()
    
    if mapa and coluna in df_gold_alunos.columns:
        coluna_destino = f"{coluna}_descricao"
        df_gold_alunos[coluna_destino] = df_gold_alunos[coluna].map(mapa)

# 4. Salvando na camada Gold
gold_dfs['alunos'] = df_gold_alunos
output_file_gold_alunos = f"{GOLD_PATH}/alunos_gold.parquet"
df_gold_alunos.to_parquet(output_file_gold_alunos, index=False)

print(f"{Fore.GREEN} Entidade ALUNOS processada com sucesso na camada Gold!")
print(f" Salvo em: {output_file_gold_alunos}")
print(f" Total de registros: {len(df_gold_alunos)}")
print(f" Colunas resultantes: {list(df_gold_alunos.columns)}\n")

 Processando consolidação Gold: Alunos

Colunas identificadas dinamicamente para tradução: ['alfabetizado', 'preenchimento_caderno', 'presenca', 'rede', 'serie']
 Entidade ALUNOS processada com sucesso na camada Gold!
 Salvo em: ./data_lake/gold/alunos_gold.parquet
 Total de registros: 3867999
 Colunas resultantes: ['ano', 'id_municipio', 'id_escola', 'id_aluno', 'caderno', 'serie', 'rede', 'presenca', 'preenchimento_caderno', 'alfabetizado', 'proficiencia', 'peso_aluno', '_ingestion_timestamp', '_source_table', '_source_system', '_record_hash', 'alfabetizado_descricao', 'preenchimento_caderno_descricao', 'presenca_descricao', 'rede_descricao', 'serie_descricao']



In [21]:
# --- 4.1 microdados de alunos: avaliados e pesos
print(f"{Fore.YELLOW} Preparando microdados de alunos para a Gold\n")

# --- só as colunas usadas
COLUNAS_ALUNOS = ['ano', 'id_municipio', 'serie', 'rede', 'presenca', 'preenchimento_caderno',
                  'alfabetizado', 'proficiencia', 'peso_aluno']
df_alunos_gold = pd.read_parquet(f"{SILVER_PATH}/alunos_silver.parquet", columns=COLUNAS_ALUNOS)
print(f"   Registros de alunos lidos: {len(df_alunos_gold):,}")


def codigo_por_descricao(coluna, contem, nao_contem=()):
    # --- acha o código pela descrição do dicionário, sem fixar '1' no código
    mapa = mapa_dicionario('alunos', coluna)
    codigos = [c for c, v in mapa.items()
               if contem in normalizar_texto(v) and not any(t in normalizar_texto(v) for t in nao_contem)]
    registrar_check(f"codigo_unico_no_dicionario:alunos.{coluna}", 'gold_microdados', 'bloqueante',
                    len(codigos) == 1, len(codigos), 1, f"dicionário: {mapa}")
    print(f"   alunos.{coluna}: código '{codigos[0]}' = '{mapa[codigos[0]]}'")
    return codigos[0]


COD_PRESENTE = codigo_por_descricao('presenca', 'present', nao_contem=('nao', 'ausent'))
COD_PREENCHIDO = codigo_por_descricao('preenchimento_caderno', 'preenchid', nao_contem=('nao',))
COD_ALFABETIZADO = codigo_por_descricao('alfabetizado', 'sim')

# --- rede e UF
checar_rede_injetiva('alunos', 'gold_microdados')
df_alunos_gold['rede_descricao'] = traduzir_codigo(df_alunos_gold['rede'], 'alunos', 'rede', 'gold_microdados')
df_alunos_gold = adicionar_chave_rede(df_alunos_gold, 'rede_descricao', 'gold_microdados')
df_alunos_gold['sigla_uf'] = df_alunos_gold['id_municipio'].str[:2].map(UF_POR_CODIGO_IBGE)
n_sem_uf = int(df_alunos_gold['sigla_uf'].isna().sum())
registrar_check('uf_derivada_do_codigo_ibge', 'gold_microdados', 'bloqueante', n_sem_uf == 0, n_sem_uf, 0)

n_antes = len(df_alunos_gold)
df_alunos_gold = df_alunos_gold[df_alunos_gold['serie'] == SERIE_ALVO]
print(f"   Filtro série {SERIE_ALVO}: {n_antes:,} -> {len(df_alunos_gold):,} registros")

# --- denominador: só quem fez a prova
df_alunos_gold['_avaliado'] = (
    (df_alunos_gold['presenca'] == COD_PRESENTE)
    & (df_alunos_gold['preenchimento_caderno'] == COD_PREENCHIDO)
    & df_alunos_gold['proficiencia'].notna()
    & df_alunos_gold['peso_aluno'].notna()
    & (df_alunos_gold['peso_aluno'] > 0)
)
df_alunos_gold['_alfabetizado'] = (df_alunos_gold['alfabetizado'] == COD_ALFABETIZADO).astype('float64')

avaliados = df_alunos_gold[df_alunos_gold['_avaliado']]
print(f"   Alunos avaliados (denominador): {len(avaliados):,} de {len(df_alunos_gold):,} registros "
      f"({len(avaliados) / len(df_alunos_gold):.1%})")

# --- taxa e pesos por município, UF e Brasil
micro_municipio = com_redes_agregadas(
    taxa_ponderada_microdado(avaliados, ['ano', 'sigla_uf', 'id_municipio', 'rede_chave']),
    ['ano', 'sigla_uf', 'id_municipio'])
micro_uf = com_redes_agregadas(
    taxa_ponderada_microdado(avaliados, ['ano', 'sigla_uf', 'rede_chave']),
    ['ano', 'sigla_uf'])
micro_brasil = com_redes_agregadas(
    taxa_ponderada_microdado(avaliados, ['ano', 'rede_chave']),
    ['ano'])

print(f"\n{Fore.GREEN} Microdados preparados.")
print(f"   Municípios x rede x ano com alunos avaliados: {len(micro_municipio):,}")
print(f"   UFs x rede x ano: {len(micro_uf):,} | Brasil x rede x ano: {len(micro_brasil):,}")

 Preparando microdados de alunos para a Gold

   Registros de alunos lidos: 3,867,999
   [BLOQUEANTE] codigo_unico_no_dicionario:alunos.presenca (gold_microdados): OK | valor=1 limiar=1 dicionário: {'0': 'Ausente', '1': 'Presente'}
   alunos.presenca: código '1' = 'Presente'
   [BLOQUEANTE] codigo_unico_no_dicionario:alunos.preenchimento_caderno (gold_microdados): OK | valor=1 limiar=1 dicionário: {'0': 'Prova não preenchida', '1': 'Prova preenchida'}
   alunos.preenchimento_caderno: código '1' = 'Prova preenchida'
   [BLOQUEANTE] codigo_unico_no_dicionario:alunos.alfabetizado (gold_microdados): OK | valor=1 limiar=1 dicionário: {'0': 'Não', '1': 'Sim'}
   alunos.alfabetizado: código '1' = 'Sim'
   Mapeamento de rede (alunos):
  codigo  descricao    rede_chave
--------  -----------  ------------
       1  Federal      federal
       2  Estadual     estadual
       3  Municipal    municipal
       4  Privada      privada
   [BLOQUEANTE] rede_chave_injetiva:alunos (gold_microdados): OK |

In [22]:
# --- 4.2 resultado vs meta por município
print(f"{Fore.YELLOW} Gold: resultado vs meta por MUNICÍPIO\n")

# --- arquivo da P3 com os eventos válidos do streaming
# (ano, id_municipio, serie, rede, taxa_alfabetizacao, event_id, event_ts, ingest_ts)
ARQUIVO_STREAMING_VALIDOS = f"{SILVER_PATH}/streaming_eventos_validos.parquet"

# --- resultado oficial
df_mun = pd.read_parquet(f"{SILVER_PATH}/municipio_silver.parquet").drop(columns=['_record_hash', '_source_system'], errors='ignore')
checar_rede_injetiva('municipio', 'gold_municipio')
df_mun['rede_descricao'] = traduzir_codigo(df_mun['rede'], 'municipio', 'rede', 'gold_municipio')
df_mun = adicionar_chave_rede(df_mun, 'rede_descricao', 'gold_municipio')
serie_desc = traduzir_codigo(df_mun['serie'], 'municipio', 'serie', 'gold_municipio')
if serie_desc is not None:
    df_mun['serie_descricao'] = serie_desc
df_mun['origem'] = 'oficial'
df_mun['fonte_taxa'] = 'municipio'

# --- eventos simulados (se a P3 já gerou)
if os.path.exists(ARQUIVO_STREAMING_VALIDOS):
    df_eventos = pd.read_parquet(ARQUIVO_STREAMING_VALIDOS)
    df_eventos = df_eventos.astype({'ano': 'Int64', 'id_municipio': 'str', 'serie': 'str', 'rede': 'str'})
    # --- repetido: fica o último ingerido
    df_eventos = (df_eventos.sort_values('ingest_ts')
                            .drop_duplicates(['ano', 'id_municipio', 'serie', 'rede'], keep='last'))
    df_eventos['rede_descricao'] = traduzir_codigo(df_eventos['rede'], 'municipio', 'rede', 'gold_municipio_simulado')
    df_eventos = adicionar_chave_rede(df_eventos, 'rede_descricao', 'gold_municipio_simulado')
    df_eventos['origem'] = 'simulado'
    df_eventos['fonte_taxa'] = 'streaming_simulado'
    print(f"   Eventos simulados incorporados: {len(df_eventos)} (origem='simulado')")
else:
    df_eventos = None
    print(f"{Fore.YELLOW}   Sem eventos simulados em {ARQUIVO_STREAMING_VALIDOS} (aguardando Pessoa 3).")

# --- meta municipal
df_meta_mun = pd.read_parquet(f"{SILVER_PATH}/meta_alfabetizacao_municipio_silver.parquet")
df_meta_mun = adicionar_chave_rede(df_meta_mun, 'rede', 'gold_municipio')
df_meta_mun = (df_meta_mun.rename(columns={'taxa_alfabetizacao': 'taxa_alfabetizacao_tabela_meta',
                                           'rede': 'rede_descricao_meta'})
                          [['ano', 'id_municipio', 'rede_chave', 'rede_descricao_meta',
                            'taxa_alfabetizacao_tabela_meta', 'nivel_alfabetizacao',
                            'percentual_participacao'] + COLUNAS_META])
checar_chave_unica(df_meta_mun, ['ano', 'id_municipio', 'rede_chave'], 'meta_alfabetizacao_municipio')

# --- mesmo recorte dos dois lados: série 2 + rede que tem meta
redes_com_meta = sorted(df_meta_mun['rede_chave'].unique())
print(f"   Redes com meta municipal: {redes_com_meta} | série alvo: {SERIE_ALVO}")


def recorte_comparavel(df):
    return df[(df['serie'] == SERIE_ALVO) & (df['rede_chave'].isin(redes_com_meta))].copy()


resultado_oficial = recorte_comparavel(df_mun)
print(f"   Resultado oficial no recorte comparável: {len(resultado_oficial):,} de {len(df_mun):,} linhas")
checar_chave_unica(resultado_oficial, ['ano', 'id_municipio', 'rede_chave'], 'municipio (recorte)')

# --- resultado x meta
partes = [montar_resultado_vs_meta(resultado_oficial, df_meta_mun, ['id_municipio'],
                                   'municipio_x_meta_municipio', 'gold_municipio', bloqueante=True)]
if df_eventos is not None:
    partes.append(montar_resultado_vs_meta(recorte_comparavel(df_eventos), df_meta_mun, ['id_municipio'],
                                           'municipio_x_meta_municipio', 'gold_municipio_simulado',
                                           bloqueante=False))
gold_municipio = pd.concat(partes, ignore_index=True)

# --- UF + peso do município (microdados)
gold_municipio['sigla_uf'] = gold_municipio['id_municipio'].str[:2].map(UF_POR_CODIGO_IBGE)
gold_municipio = join_medido(
    gold_municipio,
    micro_municipio[['ano', 'id_municipio', 'rede_chave', 'n_alunos_avaliados', 'soma_peso_avaliados']],
    ['ano', 'id_municipio', 'rede_chave'], 'municipio_x_pesos_alunos', 'gold_municipio', bloqueante=False)

# --- taxa da tabela municipio x taxa da tabela de meta
oficial = gold_municipio[gold_municipio['origem'] == 'oficial']
checar_consistencia_taxa(oficial, 'municipio', 'gold_municipio')

# --- salva
colunas_gold_municipio = (
    ['ano', 'sigla_uf', 'id_municipio', 'serie', 'rede', 'rede_descricao', 'rede_chave', 'origem',
     'fonte_taxa', 'taxa_alfabetizacao', 'media_portugues']
    + [c for c in gold_municipio.columns if c.startswith('proporcao_aluno_nivel_')]
    + ['n_alunos_avaliados', 'soma_peso_avaliados',
       'meta_do_ano', 'status_meta', 'motivo_sem_meta', 'gap_meta_do_ano_pp',
       'meta_alfabetizacao_2030', 'distancia_meta_pactuada_2030_pp', 'distancia_cenario_aspiracional_2030_pp',
       'taxa_alfabetizacao_tabela_meta', 'nivel_alfabetizacao', 'percentual_participacao']
    + [c for c in COLUNAS_META if c != 'meta_alfabetizacao_2030']
    + [c for c in ['event_id', 'event_ts', 'ingest_ts'] if c in gold_municipio.columns]
)
gold_municipio = salvar_gold(gold_municipio[colunas_gold_municipio], 'gold_municipio_resultado_vs_meta')

print(f"\n{Fore.CYAN} Situação da meta por ano (municípios, origem oficial):")
display(pd.crosstab([oficial['ano'], oficial['rede_chave']], oficial['status_meta'], margins=True))

 Gold: resultado vs meta por MUNICÍPIO



   Mapeamento de rede (municipio):
  codigo  descricao                                       rede_chave
--------  ----------------------------------------------  ----------------------------------
       0  Total (Federal, Estadual, Municipal e Privada)  total
       1  Federal                                         federal
       2  Estadual                                        estadual
       3  Municipal                                       municipal
       4  Privada                                         privada
       5  Pública (Estadual e Municipal)                  publica_estadual_municipal
       6  Pública (Federal, Estadual e Municipal)         publica_federal_estadual_municipal
   [BLOQUEANTE] rede_chave_injetiva:municipio (gold_municipio): OK | valor=0 limiar=0 
   [BLOQUEANTE] codigo_desconhecido:municipio.rede (gold_municipio): OK | valor=0 limiar=0 
   [BLOQUEANTE] rede_desconhecida (gold_municipio): OK | valor=0 limiar=0 
   [BLOQUEANTE] codigo_desconhecido:mun

,status_meta,atingiu,não atingiu,sem meta cadastrada,All
ano,rede_chave,,,,
2023,municipal,0,0,5448,5448
2024,municipal,2788,2444,216,5448
All,,2788,2444,5664,10896


In [23]:
# --- 4.3 resultado vs meta por UF
print(f"{Fore.YELLOW} Gold: resultado vs meta por UF\n")

# --- resultado oficial
df_uf = pd.read_parquet(f"{SILVER_PATH}/uf_silver.parquet").drop(columns=['_record_hash', '_source_system'], errors='ignore')
checar_rede_injetiva('uf', 'gold_uf')
df_uf['rede_descricao'] = traduzir_codigo(df_uf['rede'], 'uf', 'rede', 'gold_uf')
df_uf = adicionar_chave_rede(df_uf, 'rede_descricao', 'gold_uf')
serie_desc = traduzir_codigo(df_uf['serie'], 'uf', 'serie', 'gold_uf')
if serie_desc is not None:
    df_uf['serie_descricao'] = serie_desc
df_uf['origem'] = 'oficial'
df_uf['fonte_taxa'] = 'uf'

# --- meta por UF
df_meta_uf = pd.read_parquet(f"{SILVER_PATH}/meta_alfabetizacao_uf_silver.parquet")
df_meta_uf = adicionar_chave_rede(df_meta_uf, 'rede', 'gold_uf')
df_meta_uf = (df_meta_uf.rename(columns={'taxa_alfabetizacao': 'taxa_alfabetizacao_tabela_meta',
                                         'rede': 'rede_descricao_meta'})
                        [['ano', 'sigla_uf', 'rede_chave', 'rede_descricao_meta',
                          'taxa_alfabetizacao_tabela_meta', 'percentual_participacao'] + COLUNAS_META])
checar_chave_unica(df_meta_uf, ['ano', 'sigla_uf', 'rede_chave'], 'meta_alfabetizacao_uf')

# --- recorte: série 2 + rede pública
redes_com_meta_uf = sorted(df_meta_uf['rede_chave'].unique())
print(f"   Redes com meta estadual: {redes_com_meta_uf} | série alvo: {SERIE_ALVO}")
resultado_uf = df_uf[(df_uf['serie'] == SERIE_ALVO) & (df_uf['rede_chave'].isin(redes_com_meta_uf))].copy()
print(f"   Resultado oficial no recorte comparável: {len(resultado_uf)} de {len(df_uf)} linhas")
checar_chave_unica(resultado_uf, ['ano', 'sigla_uf', 'rede_chave'], 'uf (recorte)')

gold_uf = montar_resultado_vs_meta(resultado_uf, df_meta_uf, ['sigla_uf'],
                                   'uf_x_meta_uf', 'gold_uf', bloqueante=True)
checar_consistencia_taxa(gold_uf, 'uf', 'gold_uf')

# --- 2025 só existe na tabela de meta, a taxa vem de lá
anos_resultado_uf = set(resultado_uf['ano'].astype(int))
so_meta = df_meta_uf[~df_meta_uf['ano'].astype(int).isin(anos_resultado_uf)].copy()
if len(so_meta):
    so_meta = so_meta.assign(taxa_alfabetizacao=so_meta['taxa_alfabetizacao_tabela_meta'],
                             serie=SERIE_ALVO, rede_descricao=so_meta['rede_descricao_meta'],
                             origem='oficial', fonte_taxa='meta_alfabetizacao_uf')
    so_meta['meta_do_ano'] = extrair_meta_do_ano(so_meta)
    so_meta = classificar_meta(so_meta, np.ones(len(so_meta), dtype=bool), np.ones(len(so_meta), dtype=bool))
    gold_uf = pd.concat([gold_uf, so_meta], ignore_index=True)
    print(f"   Linhas de anos só presentes na tabela de meta: {len(so_meta)} "
          f"(anos {sorted(so_meta['ano'].astype(int).unique().tolist())}, fonte_taxa='meta_alfabetizacao_uf')")

# --- peso da UF (pra média nacional ponderada)
gold_uf = join_medido(gold_uf,
                      micro_uf[['ano', 'sigla_uf', 'rede_chave', 'n_alunos_avaliados', 'soma_peso_avaliados']],
                      ['ano', 'sigla_uf', 'rede_chave'], 'uf_x_pesos_alunos', 'gold_uf', bloqueante=False)

# --- salva
colunas_gold_uf = (
    ['ano', 'sigla_uf', 'serie', 'rede', 'rede_descricao', 'rede_chave', 'origem',
     'fonte_taxa', 'taxa_alfabetizacao', 'media_portugues']
    + [c for c in gold_uf.columns if c.startswith('proporcao_aluno_nivel_')]
    + ['n_alunos_avaliados', 'soma_peso_avaliados',
       'meta_do_ano', 'status_meta', 'motivo_sem_meta', 'gap_meta_do_ano_pp',
       'meta_alfabetizacao_2030', 'distancia_meta_pactuada_2030_pp', 'distancia_cenario_aspiracional_2030_pp',
       'taxa_alfabetizacao_tabela_meta', 'percentual_participacao']
    + [c for c in COLUNAS_META if c != 'meta_alfabetizacao_2030']
)
gold_uf = salvar_gold(gold_uf[colunas_gold_uf], 'gold_uf_resultado_vs_meta')

print(f"\n{Fore.CYAN} Situação da meta por ano (UFs):")
display(pd.crosstab([gold_uf['ano'], gold_uf['rede_chave']], gold_uf['status_meta'], margins=True))

 Gold: resultado vs meta por UF

   Mapeamento de rede (uf):
  codigo  descricao                                       rede_chave
--------  ----------------------------------------------  ----------------------------------
       0  Total (Federal, Estadual, Municipal e Privada)  total
       1  Federal                                         federal
       2  Estadual                                        estadual
       3  Municipal                                       municipal
       4  Privada                                         privada
       5  Pública (Estadual e Municipal)                  publica_estadual_municipal
       6  Pública (Federal, Estadual e Municipal)         publica_federal_estadual_municipal
   [BLOQUEANTE] rede_chave_injetiva:uf (gold_uf): OK | valor=0 limiar=0 
   [BLOQUEANTE] codigo_desconhecido:uf.rede (gold_uf): OK | valor=0 limiar=0 
   [BLOQUEANTE] rede_desconhecida (gold_uf): OK | valor=0 limiar=0 
   [BLOQUEANTE] codigo_desconhecido:uf.serie (gol

,status_meta,atingiu,não atingiu,sem meta cadastrada,All
ano,rede_chave,,,,
2023,publica_estadual_municipal,0,0,24,24
2024,publica_estadual_municipal,11,13,1,25
2025,publica_estadual_municipal,20,6,1,27
All,,31,19,26,76


In [24]:
# --- 4.4 resultado vs meta do Brasil
print(f"{Fore.YELLOW} Gold: resultado vs meta do BRASIL\n")

# --- a tabela de meta nacional já traz a taxa oficial
df_meta_br = pd.read_parquet(f"{SILVER_PATH}/meta_alfabetizacao_brasil_silver.parquet")
df_meta_br = adicionar_chave_rede(df_meta_br, 'rede', 'gold_brasil')
checar_chave_unica(df_meta_br, ['ano', 'rede_chave'], 'meta_alfabetizacao_brasil')
gold_brasil = (df_meta_br.rename(columns={'rede': 'rede_descricao'})
                         [['ano', 'rede_descricao', 'rede_chave', 'taxa_alfabetizacao',
                           'percentual_participacao'] + COLUNAS_META].copy())
gold_brasil['origem'] = 'oficial'
gold_brasil['fonte_taxa'] = 'meta_alfabetizacao_brasil'
gold_brasil['meta_do_ano'] = extrair_meta_do_ano(gold_brasil)
gold_brasil = classificar_meta(gold_brasil,
                               tem_linha_meta=np.ones(len(gold_brasil), dtype=bool),
                               ano_coberto=np.ones(len(gold_brasil), dtype=bool))

# --- UFs agregadas: ponderada (alunos avaliados) e simples, só pra comparar
uf_com_peso = gold_uf[gold_uf['soma_peso_avaliados'].notna() & gold_uf['taxa_alfabetizacao'].notna()]
agregado_ufs = (uf_com_peso.assign(_taxa_x_peso=uf_com_peso['taxa_alfabetizacao'] * uf_com_peso['soma_peso_avaliados'])
                           .groupby(['ano', 'rede_chave'])
                           .agg(n_ufs=('sigla_uf', 'nunique'),
                                _soma_taxa_x_peso=('_taxa_x_peso', 'sum'),
                                _soma_peso=('soma_peso_avaliados', 'sum'))
                           .reset_index())
agregado_ufs['taxa_ponderada_ufs'] = agregado_ufs['_soma_taxa_x_peso'] / agregado_ufs['_soma_peso']
agregado_ufs = agregado_ufs.drop(columns=['_soma_taxa_x_peso', '_soma_peso']).merge(
    gold_uf.groupby(['ano', 'rede_chave'])['taxa_alfabetizacao'].mean()
           .rename('taxa_media_simples_ufs').reset_index(),
    on=['ano', 'rede_chave'], how='left')

gold_brasil = join_medido(gold_brasil, agregado_ufs, ['ano', 'rede_chave'],
                          'brasil_x_agregado_ufs', 'gold_brasil', bloqueante=False)

# --- taxa recalculada pelos microdados
gold_brasil = join_medido(gold_brasil,
                          micro_brasil[['ano', 'rede_chave', 'n_alunos_avaliados', 'taxa_microdado']],
                          ['ano', 'rede_chave'], 'brasil_x_microdado', 'gold_brasil', bloqueante=False)

# --- salva
colunas_gold_brasil = (
    ['ano', 'rede_descricao', 'rede_chave', 'origem', 'fonte_taxa', 'taxa_alfabetizacao',
     'meta_do_ano', 'status_meta', 'motivo_sem_meta', 'gap_meta_do_ano_pp',
     'meta_alfabetizacao_2030', 'distancia_meta_pactuada_2030_pp', 'distancia_cenario_aspiracional_2030_pp',
     'taxa_microdado', 'n_alunos_avaliados', 'taxa_ponderada_ufs', 'taxa_media_simples_ufs', 'n_ufs',
     'percentual_participacao']
    + [c for c in COLUNAS_META if c != 'meta_alfabetizacao_2030']
)
gold_brasil = salvar_gold(gold_brasil[colunas_gold_brasil].sort_values(['rede_chave', 'ano']),
                          'gold_brasil_resultado_vs_meta')

print(f"\n{Fore.CYAN} Brasil — resultado vs meta pactuada:")
print(tabulate(gold_brasil[['ano', 'rede_chave', 'taxa_alfabetizacao', 'meta_do_ano', 'status_meta',
                            'meta_alfabetizacao_2030', 'distancia_meta_pactuada_2030_pp',
                            'distancia_cenario_aspiracional_2030_pp', 'taxa_ponderada_ufs',
                            'taxa_media_simples_ufs', 'taxa_microdado']],
               headers='keys', tablefmt='simple', showindex=False, floatfmt='.2f'))

 Gold: resultado vs meta do BRASIL

   [BLOQUEANTE] rede_desconhecida (gold_brasil): OK | valor=0 limiar=0 
   [BLOQUEANTE] chave_unica:ano+rede_chave (meta_alfabetizacao_brasil): OK | valor=0 limiar=0 
   [BLOQUEANTE] tipos_chaves_compativeis:brasil_x_agregado_ufs (gold_brasil): OK | valor=0 limiar=0 
   [INFORMATIVA] taxa_correspondencia_join:brasil_x_agregado_ufs (gold_brasil): FALHOU | valor=0.6667 limiar=1.0 2/3 linhas com par
   [BLOQUEANTE] tipos_chaves_compativeis:brasil_x_microdado (gold_brasil): OK | valor=0 limiar=0 
   [INFORMATIVA] taxa_correspondencia_join:brasil_x_microdado (gold_brasil): FALHOU | valor=0.6667 limiar=1.0 2/3 linhas com par
   Salvo em: ./data_lake/gold/gold_brasil_resultado_vs_meta.parquet (3 linhas, 26 colunas)

 Brasil — resultado vs meta pactuada:
  ano  rede_chave                    taxa_alfabetizacao    meta_do_ano  status_meta            meta_alfabetizacao_2030    distancia_meta_pactuada_2030_pp    distancia_cenario_aspiracional_2030_pp    taxa_pon

In [25]:
# --- 4.5 prova dos nove: taxa recalculada pelos alunos x taxa publicada
print(f"{Fore.YELLOW} Prova dos nove: microdado x indicador oficial\n")

# --- UF, todas as redes da tabela uf
uf_oficial = df_uf[df_uf['serie'] == SERIE_ALVO][['ano', 'sigla_uf', 'rede_chave', 'taxa_alfabetizacao']]
validacao_uf = join_medido(uf_oficial,
                           micro_uf[['ano', 'sigla_uf', 'rede_chave', 'n_alunos_avaliados', 'taxa_microdado']],
                           ['ano', 'sigla_uf', 'rede_chave'], 'uf_x_microdado', 'gold_validacao', bloqueante=False)
validacao_uf = validacao_uf.rename(columns={'sigla_uf': 'localidade'}).assign(nivel='uf')

# --- Brasil
validacao_br = (gold_brasil[['ano', 'rede_chave', 'taxa_alfabetizacao', 'n_alunos_avaliados', 'taxa_microdado']]
                .assign(localidade='BR', nivel='brasil'))

validacao = pd.concat([validacao_br, validacao_uf], ignore_index=True)
validacao = validacao.rename(columns={'taxa_alfabetizacao': 'taxa_oficial'})
validacao['diferenca_pp'] = validacao['taxa_microdado'] - validacao['taxa_oficial']
validacao['dentro_tolerancia'] = validacao['diferenca_pp'].abs() <= TOLERANCIA_PROVA_DOS_NOVE_PP
validacao['tolerancia_pp'] = TOLERANCIA_PROVA_DOS_NOVE_PP
validacao['origem'] = 'oficial'

comparaveis = validacao['taxa_microdado'].notna()
taxa_convergencia = float(validacao.loc[comparaveis, 'dentro_tolerancia'].mean()) if comparaveis.any() else 0.0
registrar_check('prova_dos_nove_convergencia', 'gold_validacao', 'informativa',
                taxa_convergencia >= 0.95, round(taxa_convergencia, 4), 0.95,
                f"{int(validacao.loc[comparaveis, 'dentro_tolerancia'].sum())}/{int(comparaveis.sum())} "
                f"recortes com |diferença| <= {TOLERANCIA_PROVA_DOS_NOVE_PP} p.p.")

validacao = salvar_gold(validacao[['nivel', 'localidade', 'ano', 'rede_chave', 'origem', 'taxa_oficial',
                                   'taxa_microdado', 'diferenca_pp', 'tolerancia_pp', 'dentro_tolerancia',
                                   'n_alunos_avaliados']],
                        'gold_validacao_microdado_vs_oficial')

print(f"\n{Fore.CYAN} Brasil — taxa publicada x recalculada a partir dos alunos:")
print(tabulate(validacao[validacao['nivel'] == 'brasil'][['ano', 'rede_chave', 'taxa_oficial', 'taxa_microdado',
                                                          'diferenca_pp', 'dentro_tolerancia', 'n_alunos_avaliados']],
               headers='keys', tablefmt='simple', showindex=False, floatfmt='.2f'))

print(f"\n{Fore.CYAN} Maiores diferenças por UF:")
display(validacao[validacao['nivel'] == 'uf']
        .assign(_abs=lambda d: d['diferenca_pp'].abs())
        .sort_values('_abs', ascending=False).drop(columns='_abs').head(10))

# --- só pra mostrar: o que acontece com peso/denominador errados (não vai pra gold)
publicos = df_alunos_gold[df_alunos_gold['rede_chave'].isin(REDES_AGREGADAS[REDE_PUBLICA_DAS_METAS])]
ilustracao = (publicos.groupby('ano')
              .apply(lambda g: pd.Series({
                  'correta_ponderada_avaliados': 100 * (g.loc[g['_avaliado'], 'peso_aluno'] * g.loc[g['_avaliado'], '_alfabetizado']).sum()
                                                 / g.loc[g['_avaliado'], 'peso_aluno'].sum(),
                  'errada_sem_peso_avaliados': 100 * g.loc[g['_avaliado'], '_alfabetizado'].mean(),
                  'errada_todos_registros': 100 * g['_alfabetizado'].mean(),
              }), include_groups=False)
              .reset_index()
              .merge(gold_brasil[gold_brasil['rede_chave'] == REDE_PUBLICA_DAS_METAS][['ano', 'taxa_alfabetizacao']]
                     .rename(columns={'taxa_alfabetizacao': 'oficial_publicada'}), on='ano', how='left'))
print(f"\n{Fore.CYAN} Rede pública (estadual e municipal), Brasil — efeito do denominador e do peso (%):")
print(tabulate(ilustracao, headers='keys', tablefmt='simple', showindex=False, floatfmt='.2f'))

# --- média simples x ponderada dos municípios contra a taxa da UF (rede municipal)
mun_ofc = gold_municipio[(gold_municipio['origem'] == 'oficial') & gold_municipio['soma_peso_avaliados'].notna()]
agreg_mun = (mun_ofc.assign(_tp=mun_ofc['taxa_alfabetizacao'] * mun_ofc['soma_peso_avaliados'])
                    .groupby(['ano', 'sigla_uf', 'rede_chave'])
                    .agg(taxa_media_simples_municipios=('taxa_alfabetizacao', 'mean'),
                         _tp=('_tp', 'sum'), _p=('soma_peso_avaliados', 'sum'))
                    .reset_index())
agreg_mun['taxa_ponderada_municipios'] = agreg_mun['_tp'] / agreg_mun['_p']
agreg_mun = agreg_mun.drop(columns=['_tp', '_p']).merge(
    uf_oficial.rename(columns={'taxa_alfabetizacao': 'taxa_uf_publicada'}),
    on=['ano', 'sigla_uf', 'rede_chave'], how='inner')
agreg_mun['erro_media_simples_pp'] = agreg_mun['taxa_media_simples_municipios'] - agreg_mun['taxa_uf_publicada']
agreg_mun['erro_ponderada_pp'] = agreg_mun['taxa_ponderada_municipios'] - agreg_mun['taxa_uf_publicada']
print(f"\n{Fore.CYAN} Erro médio absoluto contra a taxa publicada da UF (p.p.):")
print(agreg_mun.groupby(['ano', 'rede_chave'])[['erro_media_simples_pp', 'erro_ponderada_pp']]
               .apply(lambda d: d.abs().mean()).round(2).to_string())

 Prova dos nove: microdado x indicador oficial

   [BLOQUEANTE] tipos_chaves_compativeis:uf_x_microdado (gold_validacao): OK | valor=0 limiar=0 
   [INFORMATIVA] taxa_correspondencia_join:uf_x_microdado (gold_validacao): OK | valor=0.9793 limiar=0.95 142/145 linhas com par
   [INFORMATIVA] prova_dos_nove_convergencia (gold_validacao): OK | valor=0.9792 limiar=0.95 141/144 recortes com |diferença| <= 0.5 p.p.
   Salvo em: ./data_lake/gold/gold_validacao_microdado_vs_oficial.parquet (148 linhas, 12 colunas)

 Brasil — taxa publicada x recalculada a partir dos alunos:
  ano  rede_chave                    taxa_oficial    taxa_microdado    diferenca_pp  dentro_tolerancia      n_alunos_avaliados
-----  --------------------------  --------------  ----------------  --------------  -------------------  --------------------
 2023  publica_estadual_municipal           55.90             57.46            1.56  False                          1502809.00
 2024  publica_estadual_municipal           59.

,nivel,localidade,ano,rede_chave,origem,taxa_oficial,taxa_microdado,diferenca_pp,tolerancia_pp,dentro_tolerancia,n_alunos_avaliados,_gold_run_ts
9,uf,MT,2023,estadual,oficial,48.02,49.513061,1.493061,0.5,False,2818.0,20261031_212958
119,uf,PB,2024,estadual,oficial,54.97,55.665581,0.695581,0.5,False,1180.0,20261031_212958
13,uf,MS,2023,municipal,oficial,47.53,47.851380,0.321380,0.5,True,36442.0,20261031_212958
65,uf,MS,2023,publica_estadual_municipal,oficial,47.45,47.764934,0.314934,0.5,True,37113.0,20261031_212958
20,uf,SE,2023,publica_estadual_municipal,oficial,31.30,31.038118,-0.261882,0.5,True,18634.0,20261031_212958
42,uf,SE,2023,municipal,oficial,30.57,30.311209,-0.258791,0.5,True,15768.0,20261031_212958
30,uf,SE,2023,estadual,oficial,35.31,35.059159,-0.250841,0.5,True,2866.0,20261031_212958
31,uf,MS,2023,estadual,oficial,42.90,43.070045,0.170045,0.5,True,671.0,20261031_212958
130,uf,RS,2024,estadual,oficial,45.63,45.722804,0.092804,0.5,True,28509.0,20261031_212958
132,uf,RS,2024,publica_estadual_municipal,oficial,44.67,44.756162,0.086162,0.5,True,89562.0,20261031_212958



 Rede pública (estadual e municipal), Brasil — efeito do denominador e do peso (%):
  ano    correta_ponderada_avaliados    errada_sem_peso_avaliados    errada_todos_registros    oficial_publicada
-----  -----------------------------  ---------------------------  ------------------------  -------------------
 2023                          57.46                        58.39                     50.21                55.90
 2024                          59.20                        59.78                     52.21                59.20

 Erro médio absoluto contra a taxa publicada da UF (p.p.):
                 erro_media_simples_pp  erro_ponderada_pp
ano  rede_chave                                          
2023 municipal                    3.39               0.02
2024 municipal                    4.12               0.01


In [26]:
# --- 4.6 evolução do indicador e cenário 2030
print(f"{Fore.YELLOW} Gold: evolução do indicador e cenário 2030\n")

# --- série histórica (brasil, uf e município juntos)
colunas_evolucao = ['nivel', 'localidade', 'sigla_uf', 'ano', 'rede_chave', 'origem', 'fonte_taxa', 'taxa_alfabetizacao',
                    'meta_do_ano', 'status_meta', 'meta_alfabetizacao_2030', 'distancia_meta_pactuada_2030_pp']
evolucao = pd.concat([
    gold_brasil.assign(nivel='brasil', localidade='BR', sigla_uf=None),
    gold_uf.assign(nivel='uf', localidade=gold_uf['sigla_uf']),
    gold_municipio.assign(nivel='municipio', localidade=gold_municipio['id_municipio']),
], ignore_index=True)[colunas_evolucao]

evolucao = evolucao.sort_values(['nivel', 'localidade', 'rede_chave', 'origem', 'ano']).reset_index(drop=True)
grupo = evolucao.groupby(['nivel', 'localidade', 'rede_chave', 'origem'])
evolucao['ano_anterior'] = grupo['ano'].shift(1)
evolucao['variacao_pp_ano_anterior'] = grupo['taxa_alfabetizacao'].diff()
# --- variação só entre anos seguidos
evolucao.loc[(evolucao['ano'] - evolucao['ano_anterior']) != 1, 'variacao_pp_ano_anterior'] = np.nan
evolucao = evolucao.drop(columns='ano_anterior')
evolucao = salvar_gold(evolucao, 'gold_evolucao_indicador')


# --- reta com 2-3 pontos: é cenário, não previsão
def cenario_linear(g):
    g = g.dropna(subset=['taxa_alfabetizacao'])
    anos, taxas = g['ano'].astype(float).to_numpy(), g['taxa_alfabetizacao'].to_numpy()
    if len(g) < 2:
        inclinacao, taxa_2030 = np.nan, np.nan
    else:
        inclinacao, intercepto = np.polyfit(anos, taxas, 1)
        taxa_2030 = float(np.clip(inclinacao * 2030 + intercepto, 0, 100))
    meta_2030 = g['meta_alfabetizacao_2030'].dropna()
    meta_2030 = float(meta_2030.iloc[-1]) if len(meta_2030) else np.nan
    return pd.Series({
        'anos_observados': ', '.join(str(int(a)) for a in anos),
        'n_anos_observados': len(g),
        'ultima_taxa_observada': taxas[-1] if len(taxas) else np.nan,
        'inclinacao_pp_por_ano': inclinacao,
        'taxa_cenario_tendencia_2030': taxa_2030,
        'meta_pactuada_2030': meta_2030,
        'cenario_atinge_meta_pactuada_2030': (taxa_2030 >= meta_2030) if pd.notna(taxa_2030) and pd.notna(meta_2030) else None,
        'cenario_aspiracional_2030': CENARIO_ASPIRACIONAL_2030,
        'tipo_projecao': 'cenário de tendência linear com poucos pontos (não é previsão)',
    })


base_cenario = evolucao[(evolucao['nivel'].isin(['brasil', 'uf'])) & (evolucao['origem'] == 'oficial')]
cenario = (base_cenario.groupby(['nivel', 'localidade', 'rede_chave'])
                       .apply(cenario_linear, include_groups=False)
                       .reset_index())
cenario['origem'] = 'oficial'
cenario = salvar_gold(cenario, 'gold_cenario_2030')

print(f"\n{Fore.CYAN} Cenário Brasil até 2030 (tendência linear, não é previsão):")
print(tabulate(cenario[cenario['nivel'] == 'brasil'].drop(columns=['nivel', 'origem']),
               headers='keys', tablefmt='simple', showindex=False, floatfmt='.2f'))

 Gold: evolução do indicador e cenário 2030

   Salvo em: ./data_lake/gold/gold_evolucao_indicador.parquet (10975 linhas, 14 colunas)
   Salvo em: ./data_lake/gold/gold_cenario_2030.parquet (28 linhas, 14 colunas)

 Cenário Brasil até 2030 (tendência linear, não é previsão):
localidade    rede_chave                  anos_observados      n_anos_observados    ultima_taxa_observada    inclinacao_pp_por_ano    taxa_cenario_tendencia_2030    meta_pactuada_2030  cenario_atinge_meta_pactuada_2030      cenario_aspiracional_2030  tipo_projecao                                                      _gold_run_ts
------------  --------------------------  -----------------  -------------------  -----------------------  -----------------------  -----------------------------  --------------------  -----------------------------------  ---------------------------  --------------------------------------------------------------  ---------------
BR            publica_estadual_municipal  2023, 2024, 2025    

In [27]:
# --- 4.7 base de features pra fase 3
# alvo = taxa do município no ano t; features = só o que já se sabe antes da prova de t
print(f"{Fore.YELLOW} Gold: base de features para a Fase 3\n")

ALVO = 'alvo_taxa_alfabetizacao'

# --- colunas do mesmo ano que entregam o alvo
COLUNAS_PROIBIDAS = ['taxa_alfabetizacao', 'status_meta', 'gap_meta_do_ano_pp', 'distancia_meta_pactuada_2030_pp',
                     'distancia_cenario_aspiracional_2030_pp', 'taxa_alfabetizacao_tabela_meta',
                     'nivel_alfabetizacao', 'media_portugues', 'percentual_participacao',
                     'n_alunos_avaliados', 'soma_peso_avaliados', 'motivo_sem_meta'] + \
                    [c for c in gold_municipio.columns if c.startswith('proporcao_aluno_nivel_')]

mun = gold_municipio[gold_municipio['origem'] == 'oficial'].copy()
colunas_lag = (['taxa_alfabetizacao', 'media_portugues', 'percentual_participacao', 'n_alunos_avaliados']
               + [c for c in mun.columns if c.startswith('proporcao_aluno_nivel_')])

# --- ano anterior do município
lag = mun[['ano', 'id_municipio', 'rede_chave'] + colunas_lag].copy()
lag['ano'] = lag['ano'] + 1
lag = lag.rename(columns={c: f"{c}_ano_anterior" for c in colunas_lag})

# --- ano anterior da UF (rede pública)
lag_uf = gold_uf[gold_uf['rede_chave'] == REDE_PUBLICA_DAS_METAS][['ano', 'sigla_uf', 'taxa_alfabetizacao']].copy()
lag_uf['ano'] = lag_uf['ano'] + 1
lag_uf = lag_uf.rename(columns={'taxa_alfabetizacao': 'taxa_uf_publica_ano_anterior'})

features = (mun[['ano', 'sigla_uf', 'id_municipio', 'rede_chave', 'origem', 'taxa_alfabetizacao',
                 'meta_do_ano', 'meta_alfabetizacao_2030']]
            .merge(lag, on=['ano', 'id_municipio', 'rede_chave'], how='inner')
            .merge(lag_uf, on=['ano', 'sigla_uf'], how='left')
            .rename(columns={'taxa_alfabetizacao': ALVO,
                             'meta_do_ano': 'meta_pactuada_do_ano',
                             'meta_alfabetizacao_2030': 'meta_pactuada_2030'}))

# --- tira coluna vazia ou constante (ex.: níveis de 2023 não publicados)
sem_informacao = [c for c in features.columns
                  if c not in [ALVO, 'ano', 'origem', 'sigla_uf', 'id_municipio', 'rede_chave']
                  and features[c].nunique(dropna=True) <= 1]
if sem_informacao:
    print(f"{Fore.YELLOW}   Colunas removidas por não terem informação (nulas ou constantes): {sem_informacao}")
    features = features.drop(columns=sem_informacao)

# --- vazamento de alvo
vazando = [c for c in features.columns if c in COLUNAS_PROIBIDAS]
registrar_check('sem_vazamento_de_alvo', 'gold_features_fase3', 'bloqueante', not vazando, len(vazando), 0,
                f"colunas proibidas presentes: {vazando}" if vazando else '')

# --- correlação muito alta com o alvo costuma ser vazamento
numericas = features.select_dtypes('number').drop(columns=[ALVO, 'ano'])
correlacoes = numericas.corrwith(features[ALVO]).abs().sort_values(ascending=False)
maior = correlacoes.dropna()
registrar_check('correlacao_maxima_feature_alvo', 'gold_features_fase3', 'informativa',
                maior.empty or maior.iloc[0] < 0.98,
                round(float(maior.iloc[0]), 4) if not maior.empty else None, 0.98,
                f"feature: {maior.index[0]}" if not maior.empty else '')

features = salvar_gold(features, 'gold_features_fase3')
print(f"\n   Anos-alvo disponíveis: {sorted(features['ano'].unique().tolist())}")
print(f"   Correlação absoluta de cada feature com o alvo:\n{correlacoes.round(3).to_string()}")

 Gold: base de features para a Fase 3

   Colunas removidas por não terem informação (nulas ou constantes): ['meta_pactuada_2030', 'proporcao_aluno_nivel_0_ano_anterior', 'proporcao_aluno_nivel_1_ano_anterior', 'proporcao_aluno_nivel_2_ano_anterior', 'proporcao_aluno_nivel_3_ano_anterior', 'proporcao_aluno_nivel_4_ano_anterior', 'proporcao_aluno_nivel_5_ano_anterior', 'proporcao_aluno_nivel_6_ano_anterior', 'proporcao_aluno_nivel_7_ano_anterior', 'proporcao_aluno_nivel_8_ano_anterior']
   [BLOQUEANTE] sem_vazamento_de_alvo (gold_features_fase3): OK | valor=0 limiar=0 
   [INFORMATIVA] correlacao_maxima_feature_alvo (gold_features_fase3): OK | valor=0.6506 limiar=0.98 feature: media_portugues_ano_anterior
   Salvo em: ./data_lake/gold/gold_features_fase3.parquet (5396 linhas, 13 colunas)

   Anos-alvo disponíveis: [2024]
   Correlação absoluta de cada feature com o alvo:
media_portugues_ano_anterior            0.651
taxa_alfabetizacao_ano_anterior         0.636
meta_pactuada_do_ano     

In [28]:
# --- 4.8 anomalias
import json

print(f"{Fore.YELLOW} Investigação de anomalias\n")
achados = []


def achado(codigo, descricao, evidencia):
    # --- converte pandas pra tipo simples, pra ficar legível no csv
    if isinstance(evidencia, (pd.DataFrame, pd.Series)):
        evidencia = json.loads(evidencia.to_json())
    elif isinstance(evidencia, dict):
        evidencia = {k: json.loads(v.to_json()) if isinstance(v, (pd.DataFrame, pd.Series)) else v
                     for k, v in evidencia.items()}
    achados.append({'gold_run_ts': GOLD_RUN_TS, 'codigo': codigo, 'descricao': descricao, 'evidencia': str(evidencia)})
    print(f"{Fore.CYAN} [{codigo}] {descricao}\n{Style.RESET_ALL}   {evidencia}\n")


# --- A1 trajetória da meta muda entre anos de pactuação?
for nome, df_meta, chaves in [('brasil', df_meta_br, ['rede_chave']),
                              ('uf', df_meta_uf, ['sigla_uf', 'rede_chave'])]:
    revisoes = []
    for col in COLUNAS_META:
        variacao = df_meta.groupby(chaves)[col].agg(lambda s: s.dropna().nunique())
        revisoes.append(f"{col}: {int((variacao > 1).sum())} localidade(s) com valor diferente entre linhas de ano")
    achado(f'A1_{nome}', f'Trajetória da meta revisada entre anos de pactuação ({nome})', ' | '.join(revisoes))
achado('A1_brasil_valores', 'Trajetória nacional por ano de pactuação',
       df_meta_br.sort_values('ano')[['ano'] + COLUNAS_META].to_dict('records'))

# --- A2 cobertura de anos: resultado x meta
for nome, res, meta, chave in [('municipio', resultado_oficial, df_meta_mun, 'id_municipio'),
                               ('uf', resultado_uf, df_meta_uf, 'sigla_uf')]:
    achado(f'A2_{nome}', f'Cobertura por ano: resultado x meta ({nome})',
           {'linhas_resultado_por_ano': res.groupby('ano').size(),
            'linhas_meta_por_ano': meta.groupby('ano').size()})

# --- A3 meta municipal sem taxa / sem meta 2024
nulos_meta_mun = df_meta_mun.assign(
    taxa_nula=df_meta_mun['taxa_alfabetizacao_tabela_meta'].isna(),
    meta_2024_nula=df_meta_mun['meta_alfabetizacao_2024'].isna()
).groupby('ano')[['taxa_nula', 'meta_2024_nula']].sum()
achado('A3', 'Nulos na tabela de meta municipal por ano', nulos_meta_mun)

# --- A4 níveis de proficiência nulos por ano
nivel_nulo = df_mun.assign(nivel_nulo=df_mun['proporcao_aluno_nivel_0'].isna()).groupby('ano')['nivel_nulo'].agg(['sum', 'size'])
achado('A4', 'proporcao_aluno_nivel_* nulo por ano (municipio)', nivel_nulo)

# --- A5 flag alfabetizado bate com o corte de 743?
av = df_alunos_gold[df_alunos_gold['_avaliado']]
pelo_corte = (av['proficiencia'] >= PONTO_CORTE_SAEB).astype('float64')
divergencia = (av.assign(_div=(pelo_corte != av['_alfabetizado']))
                 .groupby('ano')['_div'].agg(['sum', 'mean']))
achado('A5', f"Flag 'alfabetizado' x proficiência >= {PONTO_CORTE_SAEB} (avaliados)", divergencia.round(4))

# --- A6 presença x proficiência
sem_prof = df_alunos_gold[(df_alunos_gold['presenca'] == COD_PRESENTE)
                          & (df_alunos_gold['preenchimento_caderno'] == COD_PREENCHIDO)
                          & df_alunos_gold['proficiencia'].isna()]
prof_sem_presenca = df_alunos_gold[(df_alunos_gold['presenca'] != COD_PRESENTE) & df_alunos_gold['proficiencia'].notna()]
achado('A6', 'Inconsistências entre presença, preenchimento e proficiência',
       {'presente_preenchido_sem_proficiencia': len(sem_prof), 'proficiencia_sem_presenca': len(prof_sem_presenca)})

# --- A7 UF sem meta
achado('A7', 'UFs com meta 2030 nula na fonte',
       df_meta_uf[df_meta_uf['meta_alfabetizacao_2030'].isna()][['ano', 'sigla_uf']].to_dict('records'))

# --- A8 prova dos nove fora da tolerância
fora = validacao[~validacao['dentro_tolerancia'] & validacao['taxa_microdado'].notna()]
achado('A8', f'Recortes com |microdado - oficial| > {TOLERANCIA_PROVA_DOS_NOVE_PP} p.p.',
       fora[['nivel', 'localidade', 'ano', 'rede_chave', 'diferenca_pp']].round(2).to_dict('records')[:15])

# --- A9 taxa nacional publicada x UFs ponderadas
nacional = gold_brasil[['ano', 'taxa_alfabetizacao', 'taxa_ponderada_ufs', 'taxa_microdado']].copy()
nacional['diferenca_publicada_vs_ufs_pp'] = (nacional['taxa_alfabetizacao'] - nacional['taxa_ponderada_ufs']).round(2)
achado('A9', 'Taxa nacional publicada x média ponderada das UFs publicadas', nacional.round(2).to_dict('records'))

df_achados = pd.DataFrame(achados)
df_achados.to_csv(ARQUIVO_ANOMALIAS, mode='a', index=False, header=not os.path.exists(ARQUIVO_ANOMALIAS))
print(f"{Fore.GREEN} {len(df_achados)} diagnósticos gravados em {ARQUIVO_ANOMALIAS}")

 Investigação de anomalias

 [A1_brasil] Trajetória da meta revisada entre anos de pactuação (brasil)
   meta_alfabetizacao_2024: 1 localidade(s) com valor diferente entre linhas de ano | meta_alfabetizacao_2025: 1 localidade(s) com valor diferente entre linhas de ano | meta_alfabetizacao_2026: 1 localidade(s) com valor diferente entre linhas de ano | meta_alfabetizacao_2027: 1 localidade(s) com valor diferente entre linhas de ano | meta_alfabetizacao_2028: 1 localidade(s) com valor diferente entre linhas de ano | meta_alfabetizacao_2029: 1 localidade(s) com valor diferente entre linhas de ano | meta_alfabetizacao_2030: 0 localidade(s) com valor diferente entre linhas de ano

 [A1_uf] Trajetória da meta revisada entre anos de pactuação (uf)
   meta_alfabetizacao_2024: 22 localidade(s) com valor diferente entre linhas de ano | meta_alfabetizacao_2025: 23 localidade(s) com valor diferente entre linhas de ano | meta_alfabetizacao_2026: 20 localidade(s) com valor diferente entre linhas de 

In [29]:
# --- 4.9 números pro vídeo (tudo que for citado sai daqui)
print(f"{Fore.YELLOW} Números de referência para o vídeo\n")
numeros = []


def numero(indicador, valor, unidade, fonte):
    numeros.append({'indicador': indicador, 'valor': valor, 'unidade': unidade, 'tabela_gold': fonte})


# --- brasil
br_pub = gold_brasil[gold_brasil['rede_chave'] == REDE_PUBLICA_DAS_METAS].sort_values('ano')
for _, r in br_pub.iterrows():
    a = int(r['ano'])
    numero(f'Taxa de alfabetização Brasil, rede pública, {a}', round(r['taxa_alfabetizacao'], 2), '%', 'gold_brasil_resultado_vs_meta')
    numero(f'Meta pactuada Brasil para {a}', None if pd.isna(r['meta_do_ano']) else round(r['meta_do_ano'], 2), '%', 'gold_brasil_resultado_vs_meta')
    numero(f'Situação da meta Brasil {a}', r['status_meta'], '-', 'gold_brasil_resultado_vs_meta')
    numero(f'Taxa recalculada pelos microdados, {a}',
           None if pd.isna(r['taxa_microdado']) else round(r['taxa_microdado'], 2), '%', 'gold_brasil_resultado_vs_meta')
if len(br_pub):
    ult = br_pub.iloc[-1]
    numero('Meta pactuada 2030 (rede pública, fonte)', ult['meta_alfabetizacao_2030'], '%', 'gold_brasil_resultado_vs_meta')
    numero(f"Distância até a meta pactuada 2030 a partir de {int(ult['ano'])}",
           round(ult['distancia_meta_pactuada_2030_pp'], 2), 'p.p.', 'gold_brasil_resultado_vs_meta')
    numero(f"Distância até o cenário aspiracional (100%) a partir de {int(ult['ano'])}",
           round(ult['distancia_cenario_aspiracional_2030_pp'], 2), 'p.p.', 'gold_brasil_resultado_vs_meta')

cen_br = cenario[(cenario['nivel'] == 'brasil') & (cenario['rede_chave'] == REDE_PUBLICA_DAS_METAS)]
if len(cen_br):
    numero('Cenário de tendência linear Brasil 2030 (não é previsão)',
           round(cen_br['taxa_cenario_tendencia_2030'].iloc[0], 2), '%', 'gold_cenario_2030')

# --- municípios
mun_ofc = gold_municipio[gold_municipio['origem'] == 'oficial']
ultimo_ano_meta = mun_ofc.loc[mun_ofc['meta_do_ano'].notna(), 'ano'].max()
if pd.notna(ultimo_ano_meta):
    contagem = mun_ofc[mun_ofc['ano'] == ultimo_ano_meta]['status_meta'].value_counts()
    for status, qtd in contagem.items():
        numero(f'Municípios ({int(ultimo_ano_meta)}): {status}', int(qtd), 'municípios', 'gold_municipio_resultado_vs_meta')

# --- UFs (maior e menor)
ufs_ult = gold_uf[gold_uf['ano'] == gold_uf['ano'].max()].dropna(subset=['taxa_alfabetizacao'])
if len(ufs_ult):
    maior, menor = ufs_ult.loc[ufs_ult['taxa_alfabetizacao'].idxmax()], ufs_ult.loc[ufs_ult['taxa_alfabetizacao'].idxmin()]
    numero(f"Maior taxa entre UFs ({int(maior['ano'])}): {maior['sigla_uf']}", round(maior['taxa_alfabetizacao'], 2), '%', 'gold_uf_resultado_vs_meta')
    numero(f"Menor taxa entre UFs ({int(menor['ano'])}): {menor['sigla_uf']}", round(menor['taxa_alfabetizacao'], 2), '%', 'gold_uf_resultado_vs_meta')

# --- salva
df_numeros = pd.DataFrame(numeros)
df_numeros['gold_run_ts'] = GOLD_RUN_TS
df_numeros.to_csv(f"{GOLD_PATH}/gold_numeros_video.csv", index=False)
print(tabulate(df_numeros.drop(columns='gold_run_ts'), headers='keys', tablefmt='simple', showindex=False))
print(f"\n{Fore.GREEN} Salvo em: {GOLD_PATH}/gold_numeros_video.csv")

 Números de referência para o vídeo

indicador                                                     valor                unidade     tabela_gold
------------------------------------------------------------  -------------------  ----------  --------------------------------
Taxa de alfabetização Brasil, rede pública, 2023              55.9                 %           gold_brasil_resultado_vs_meta
Meta pactuada Brasil para 2023                                                     %           gold_brasil_resultado_vs_meta
Situação da meta Brasil 2023                                  sem meta cadastrada  -           gold_brasil_resultado_vs_meta
Taxa recalculada pelos microdados, 2023                       57.46                %           gold_brasil_resultado_vs_meta
Taxa de alfabetização Brasil, rede pública, 2024              59.2                 %           gold_brasil_resultado_vs_meta
Meta pactuada Brasil para 2024                                59.9                 %           gold_bra

## Resultados e anomalias

Diagnósticos em `quality_reports/gold_anomalias.csv` e regras em `quality_reports/gold_checks.csv`.

**Prova dos nove.** 141 de 144 recortes (UF x rede x ano) recalculados a partir dos alunos ficaram a até 0,5 p.p. da taxa publicada. Brasil 2024 bateu exato (59,2%). A média ponderada dos municípios erra 0,01–0,02 p.p. contra a taxa da UF; a média simples erra 3,4–4,1 p.p. Usando todos os registros no denominador (com ausentes), a taxa de 2024 cairia de 59,2% para 52,2%.

**Anomalias encontradas:**

- **Duas redes "públicas".** O dicionário tem a rede 5 (estadual e municipal) e a 6 (federal, estadual e municipal), e as metas dizem só "Pública". Pelos dados é a 5: a taxa da meta por UF bate com ela em 49 de 49 casos. A rede 6 não aparece em nenhuma linha.
- **A meta muda conforme o ano de pactuação.** A linha de 2025 da meta nacional traz 60 / 64 / 67... e as de 2023/2024 traziam 59,9 / 63,77 / 67,47... Nas UFs, 20 a 25 estados tiveram revisão. A meta de 2030 não mudou. Por isso a meta de cada ano é lida da linha daquele ano.
- **2025 só existe nas tabelas de meta.** As tabelas `uf` e `municipio` vão até 2024 e não há microdado de 2025. A taxa de 2025 das UFs entra a partir da meta (`fonte_taxa = meta_alfabetizacao_uf`) e fica fora da prova dos nove.
- **Brasil 2023 não fecha com as UFs.** O publicado é 55,9%, mas a média ponderada das UFs e o microdado dão 57,46%. Cada UF bate individualmente, e em 2024 a diferença é 0,01 p.p. Suspeitamos de outro método na linha de base nacional, mas não conseguimos confirmar. Mantivemos o número oficial.
- **Níveis de proficiência só a partir de 2024.** `proporcao_aluno_nivel_*` está vazio em todas as linhas de 2023.
- **120 municípios sem meta de 2024** em cada ano, e 120 linhas sem taxa em 2023. Provavelmente municípios sem resultado na linha de base (não confirmado um a um). Ficam como "sem meta cadastrada".
- **Roraima sem meta estadual** em 2023 e 2024.
- **13,3% dos alunos sem proficiência** (513.338 de 3.867.999): são ausentes ou caderno não preenchido, e ficam fora do denominador. Não achamos presente sem nota nem nota sem presença.
- **3 municípios** com taxa diferente entre a tabela de resultado e a de meta (até 36 p.p.). Erro pontual da fonte, registrado.

## Uso em IA (Fase 3)

`gold_features_fase3`: uma linha por município, rede municipal. O alvo é a taxa do ano t. As features são o que já se sabia antes da prova: taxa, proficiência média, participação e avaliados do ano anterior, taxa da UF no ano anterior e meta do ano. Ficaram de fora as colunas do mesmo ano que vêm da mesma prova (proficiência, níveis, status da meta etc.), e uma regra bloqueante garante isso. A maior correlação de uma feature com o alvo é 0,65.

Limitação: como os resultados vão só até 2024, há um único ano-alvo (2024). Com um ano novo dá para validar treinando em t e testando em t+1.

Usos possíveis:
- prever quais municípios tendem a não bater a meta do ano seguinte e priorizar apoio antes da avaliação (em 2024, 2.444 não bateram e 2.788 bateram)
- medir a desigualdade entre e dentro das UFs ao longo do tempo (em 2025 a taxa estadual vai de 48% no RN a 84% no CE)
- separar quem precisa de apoio (não atingiu) de quem primeiro precisa ter a meta cadastrada (216 municípios em 2024)

**Para o vídeo:** usar os números de `gold_numeros_video.csv`. O cenário de 90,7% em 2030 sai de 3 pontos e é puxado pelo salto de 2024 para 2025, então deve ser apresentado como cenário. Se citar Brasil 2023, dizer que ele não fecha com as UFs.